# Korea FCFF Individual Valuation **v1.5** (base: v8 모형)

개별 종목 리스트를 입력하면 종목별 FCFF DCF 평가 후 **9-sheet Excel** 을 저장하는 노트북.

- 모형 로직·Excel 저장 방식·저장 내용은 `korea_fcff_dcf_valuation_v8` 과 동일 (US v13 Validity 가드 포함)
- **모든 입력 변수는 Cell 2 한 곳**에 집중 — Cell 2 만 수정하고 나머지는 위→아래 순서대로 실행
- 개별 진단용 시각화(Decomposition/Sensitivity)는 제거 → run → Excel 저장의 단순 흐름
- **v1.1**: Excel 파일명 끝에 FCFF 측정일자(`_YYYYMMDD`) 자동 부착
- **v1.5**: ★ **OPM 추정 개편 + Phase 2 미드사이클 수렴**
  - 구 v1.4 는 OPM 시계열만 따로 예측한 뒤 **[-30%, 50%] 고정 clip** → 호황기(예: SK하이닉스 OPM 76%)엔 8분기 전부 50% 로 잘려, 매출이 늘어도 EBIT 가 급감하는 비정합 발생
  - `OPM_MODE="regression"`(기본): **매출-영업이익 회귀**(level: OP=α+β·매출+분기더미 / diff: YoY 증감 / log) 를 최근 4Q 홀드아웃 MAE 로 자동 선택 → 매출 예측과 연동된 OPM. 고정 50% 상한 폐지 → [창 내 최저−10%p, 최고+10%p]
  - `MIDCYCLE_ENABLE=True`(기본): Phase 1 말 OPM → 과거 10년 미드사이클 OPM 으로 Phase 2 동안 선형 수렴 (FCFF += 매출경로 × ΔOPM × (1−T)) → 정점 마진이 TV 로 영구화되는 것을 차단
  - **OPM_Model 시트** 신규(회귀식·홀드아웃·TS 비교·미드사이클 경로·TP 정합성) + Phase2_LongTerm 동기화 → **11-sheet**
- **v1.4**: ★ **forecast 로드 ticker 형식 혼재 흡수** — `korea_revenue_forecast_result` 에 'A003350'(구버전)·'003350'(v5+ 표준) 형식이 섞여 있어도 두 형식을 함께 조회해 최신 실행 버전을 로드 (한 형식만 보면 stale 구버전 예측을 잡던 문제 수정)
- **v1.3**: ★ **forecast↔actual 시점 정렬 가드** — DB 매출예측이 구버전(예: 2025Q3 기준)인데 actual 이 더 최신(2026Q2)이면, actual 과 겹치는 stale forecast 분기를 제거하고 actual+1Q 부터 이어 붙임 (불연속·전량중복은 raise, 잔여 horizon 부족은 `ALLOW_SHORT_FORECAST` 토글). ΔNWC 시드 왜곡도 함께 해소
- **v1.2**: ★ **NWC_Detail 시트** 추가 (US `FCFF_DCF_Individual_v16_2` 이식) — 운전자본 구성(매출채권·재고·매입채무) + 매출TTM·COGS TTM·**DSO/DIO/DPO/CCC** 회전일수 시계열 + 최근 8Q 안정성(CV) 진단. 9-sheet → **10-sheet** Excel

## Cell 1 · 경로 자동 감지 (노트북/데스크탑 공용)

In [13]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  프로젝트 루트(= DATA 폴더의 부모)를 sys.path 에 등록.
#  → 노트북/데스크탑 어느 PC 에서 실행해도
#     from DATA.config import ...
#     from DATA.us_target_ticker_list_screened_20260411 import ...
#     처럼 DATA.* 모듈이 동일하게 import 됩니다.
#  ▸ 노트북   : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA
#  ▸ 데스크탑 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA
# ─────────────────────────────────────────────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로    : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


## Cell 2 · ★ 입력 변수 (여기만 수정하세요)

- `EXPORT_TICKERS` : 평가할 종목 리스트
- `EXPORT_DIR` : Excel 저장 폴더
- `SAVE_TO_DB` : DB 동시 저장 여부 (기본 False = Excel 만)
- 이하 모델 파라미터·v8 가드 상수 (기본값 = v8 과 동일)

In [14]:
# ═══════════════════════════════════════════════════════════════
#  ★★★ 입력 변수 셀 — 이 노트북의 모든 사용자 입력은 여기 한 곳 ★★★
#  (아래 셀들은 수정할 필요가 없습니다 — 실행만 하면 됩니다)
# ═══════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────
#  ① 평가 대상 종목 & 출력
# ─────────────────────────────────────────────────────────────
EXPORT_TICKERS = [          # ★ 평가할 종목 리스트 (A+6자리, 여러 개 가능)
    "A000660",              #   삼성전자 (예시 — 원하는 종목으로 교체)
    # "A000660",            #   SK하이닉스
]

EXPORT_DIR = (Path(r"C:/reports") if os.name == "nt"
              else Path.home() / "reports")   # Excel 저장 폴더 (없으면 자동 생성)

SAVE_TO_DB = False          # True → Excel 출력과 함께 DB(TABLE_RESULT)에도 저장
VERBOSE    = True           # True → 종목별 상세 계산 로그 출력

# ─────────────────────────────────────────────────────────────
#  ② DB 테이블 (변경 시에만 수정)
# ─────────────────────────────────────────────────────────────
TABLE_FS         = "korea_fs_data_from_DG"            # 재무제표 원본 (DataGuide)
TABLE_FORECAST   = "korea_revenue_forecast_result"    # 매출 예측 결과
TABLE_MARKETCAP  = "ks_listed_company_daily_marketcap"# 일별 시가총액
TABLE_PRICE      = "KSE_Price"                        # 일별 주가
TABLE_RESULT     = "korea_fcff_dcf_valuation_v8"      # SAVE_TO_DB=True 일 때 저장 테이블
TABLE_QUALITY    = "korea_valuation_quality_log"      # 데이터 품질 로그 (개별 모드에선 미사용)

# ─────────────────────────────────────────────────────────────
#  ③ 모델 파라미터
# ─────────────────────────────────────────────────────────────
FORECAST_HORIZON     = 8            # Phase 1 예측 분기 수 (8분기 = 2년)
ALLOW_SHORT_FORECAST = True         # ★ v1.3: stale forecast 정렬 후 남은 분기가
                                    #   horizon 미만이어도 축소 horizon 으로 진행 (경고)
                                    #   False → ValueError 로 차단 (재예측 강제)
MIN_HISTORY          = 16           # OPM 추정(회귀·시계열 공통)에 필요한 최소 분기 수
WINSORIZE_LIMITS     = (0.05, 0.95) # 비율 계수 추정 시 winsorize 범위
GDP_GROWTH           = 0.04         # 한국 장기 GDP 성장률 추정 (terminal g 상한)

# Phase 2 성장률(g) 허용 범위 (US v9.2 패치)
G_FLOOR_MIN          = -0.20        # 음수 g₀ 하한 (사이클릭 침체 -20% 까지 허용)
G_CEIL               = 1.00         # ★ v8: g 상한 (US v13 정렬, 구 0.50)
                                    #   적응형 ρ 상한·TV 배수 상한이 받쳐주므로 상향 안전

# ─────────────────────────────────────────────────────────────
#  ③-2 ★ v1.5 OPM 예측 (Phase 1, 8분기)
# ─────────────────────────────────────────────────────────────
OPM_MODE          = "regression"  # "regression": 매출-영업이익 회귀 (표본 부족 시에만 ts)
                                  # "auto"      : 회귀 R² ≥ OPM_REG_MIN_R2 일 때만 회귀, 아니면 ts
                                  # "ts"        : 구 v1.4 OPM 시계열(SARIMA/ETS/Theta) 방식
OPM_REG_METHOD    = "auto"        # "auto"(홀드아웃 MAE 최소 선택) | "level" | "diff" | "log"
                                  #   level: OP = α + β·매출 + 분기더미 (α<0 고정비, β 공헌이익률)
                                  #   diff : ΔOP(YoY) = α + β·Δ매출(YoY)  /  log: 전 분기 흑자일 때만
OPM_REG_WINDOW_Q  = 24            # 회귀 추정 최근 분기 수 (구조변화 반영)
OPM_HOLDOUT_Q     = 4             # auto 선택용 홀드아웃 분기 수
OPM_REG_MIN_R2    = 0.30          # OPM_MODE="auto" 일 때 회귀 채택 최소 R²
OPM_CLIP_MARGIN   = 0.10          # 예측 OPM ∈ [창 내 최저 − 10%p, 최고 + 10%p]  (구 고정 [-30%, 50%] 폐지)
OPM_ABS_BOUNDS    = (-1.00, 0.95) # 절대 안전범위

# ─────────────────────────────────────────────────────────────
#  ③-3 ★ v1.5 Phase 2 미드사이클 OPM 수렴
# ─────────────────────────────────────────────────────────────
MIDCYCLE_ENABLE    = True         # Phase 1 말 OPM → 미드사이클 OPM 으로 Phase 2 동안 수렴
MIDCYCLE_WINDOW_Q  = 40           # 미드사이클 산정 기간 (최근 40분기 = 10년, 한 사이클 이상)
MIDCYCLE_STAT      = "median"     # "median": 분기 OPM 중앙값 | "weighted": Σ영업이익 / Σ매출
MIDCYCLE_YEARS     = None         # 수렴 기간(년). None → Phase 2 전체 기간에 걸쳐 선형 수렴
MIDCYCLE_DIRECTION = "both"       # "both": 고마진↓·저마진↑ 모두 정규화 | "down_only": 고마진만 하향
MIDCYCLE_MIN_GAP   = 0.01         # |Phase1 말 OPM − 미드사이클| 이 이보다 작으면 미적용

# OLS fallback 기준 (비율 계수 회귀가 이 기준 미달이면 median fallback)
OLS_MIN_R2           = 0.30         # 회귀 채택 최소 R²
OLS_MIN_SAMPLES      = 16           # 회귀 채택 최소 표본 수

# ─────────────────────────────────────────────────────────────
#  ④ WACC / ERP
# ─────────────────────────────────────────────────────────────
ERP_METHOD           = "damodaran_floor"  # 'damodaran_floor' | 'kospi_geo_10y' | 'kospi_geo_no_floor'
DAMODARAN_ERP_KR     = 0.07               # Damodaran 한국 ERP
GEO_FLOOR            = 0.07               # KOSPI geo 사용 시 E(Rm) floor
RF_FALLBACK          = 0.035              # BOK API 실패 시 fallback 무위험이자율
RD_DEFAULT           = 0.045              # 부채비용(Rd) 산출 실패 시 기본값
WACC_FLOOR           = 0.05               # WACC 절대 하한 (v8 은 max(이 값, Rf+1%) 사용)
WACC_CAP             = 0.20               # WACC 상한

# ─────────────────────────────────────────────────────────────
#  ⑤ ★ v8 Validity 가드 (미국 v13 이식, 통화가드 제외)
# ─────────────────────────────────────────────────────────────
V8_RD_FLOOR_RF      = True     # ② Rd 하한 = Rf (forward-looking 부채비용)
V8_WACC_FLOOR_ABS   = 0.05     # ② WACC 절대 하한 → max(이 값, Rf+1%)
V8_MIN_TV_SPREAD    = 0.02     # ③ WACC − g_term 최소 스프레드 (구 0.01)
V8_TERMINAL_RF_CAP  = True     # ③ g_term ≤ min(GDP, Rf)  (Damodaran)
V8_TV_FCFF_MULT_CAP = 35.0     # ③ TV / 마지막 연간 FCFF 배수 상한
V8_SANITY_GUARD     = True     # ④ 지분가치/시총 괴리 검사
V8_SANITY_MC_RATIO  = 30.0     # ④ 허용 배율 (초과 시 평가 제외)
V8_RHO_ADAPTIVE_CAP = True     # ⑤ g0 기반 ρ 적응형 상한

# ─────────────────────────────────────────────────────────────
#  ⑥ Net Debt / NWC 구성 항목 (BS 컬럼 키)
# ─────────────────────────────────────────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds",
             "long_term_debt", "lease_liab"]           # 총부채 합산 항목
CASH_KEYS = ["cash", "short_term_invest"]              # 현금성 합산 항목

# ★ v7 비현금 영업운전자본 (Damodaran 표준)
#   NWC = (매출채권+재고자산+선급비용) − (매입채무+미지급비용+미지급금+선수금+계약부채)
USE_OPERATING_NWC = True   # False → 구 v6 proxy 로 회귀 (A/B 비교용)

# FCFF_RIM_DATA_2026_1Q 에 추가된 BS 영업운전자본 항목 (참조용)
#   (helper 의 DG_ITEM_CODES 매핑 + DB 적재가 선행되어야 함)
V7_NEW_ITEM_CODES = {
    "M000902006": "payables",              # 매입채무
    "M001122290": "accrued_expenses",      # 미지급비용
    "M001122140": "other_payables",        # 미지급금
    "M001122580": "advances_received",     # 선수금
    "M001122561": "contract_liabilities",  # 계약부채
    "M001113970": "prepaid_expenses",      # 선급비용 (영업유동자산)
}

# ─────────────────────────────────────────────────────────────
#  ⑦ 단위 환산 (수정 불필요)
# ─────────────────────────────────────────────────────────────
MARKETCAP_UNIT_MULTIPLIER = 1_000_000  # 시가총액: 백만원 → 원
FS_UNIT_MULTIPLIER        = 1_000      # 재무제표: 천원 → 원

# ── 배치 노트북과의 호환용 상수 (개별 모드에선 사실상 미사용) ──
MIN_REVENUE_QUARTERS = 24              # universe 필터 기준 (개별 모드 미사용)

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("[OK] 입력 변수 설정 완료")
print(f"  평가 대상 종목   : {EXPORT_TICKERS}")
print(f"  Excel 저장 폴더  : {EXPORT_DIR}")
print(f"  DB 저장          : {SAVE_TO_DB}  (테이블: {TABLE_RESULT})")
print(f"  Phase2 g 범위    : [{G_FLOOR_MIN:.0%}, {G_CEIL:.0%}]  GDP={GDP_GROWTH:.0%}")
print(f"  OPM 예측         : {OPM_MODE} (회귀방식={OPM_REG_METHOD}, 창={OPM_REG_WINDOW_Q}Q, clip ±{OPM_CLIP_MARGIN:.0%}p)")
print(f"  미드사이클 수렴  : {MIDCYCLE_ENABLE} ({MIDCYCLE_STAT}, {MIDCYCLE_WINDOW_Q}Q, 방향={MIDCYCLE_DIRECTION})")


[OK] 입력 변수 설정 완료
  평가 대상 종목   : ['A000660']
  Excel 저장 폴더  : C:\reports
  DB 저장          : False  (테이블: korea_fcff_dcf_valuation_v8)
  Phase2 g 범위    : [-20%, 100%]  GDP=4%
  OPM 예측         : regression (회귀방식=auto, 창=24Q, clip ±10%p)
  미드사이클 수렴  : True (median, 40Q, 방향=both)


## Cell 3 · Import & DB 연결

In [15]:
# ═══════════════════════════════════════════════════════════════
#  Import & DB 연결 — 사용자 입력 없음 (실행만 하면 됩니다)
# ═══════════════════════════════════════════════════════════════

# ── 표준 라이브러리 ──────────────────────────────────────────────
import gc, math, time, traceback, warnings
from datetime import datetime, date, timedelta
from typing import Optional, Dict, Any, List, Tuple

# ── 외부 라이브러리 ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

# ── 내부 모듈 (Cell 1 경로 자동 감지 덕분에 두 PC 모두 동일하게 import) ──
from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    setup_project_path, to_dg_ticker, to_price_ticker, get_pymysql_conn,
    DG_ITEM_CODES, get_item_code,
    load_korea_financials_wide, load_korea_revenue_forecast,
    load_korea_marketcap_latest, load_korea_price_series, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db, get_evaluation_history,
)
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima, forecast_ets, forecast_theta,
    infer_freq_alias, seasonal_periods_from_freq, clear_memory,
)
# (필요 시) US 스크리닝 티커 리스트도 동일 방식으로 import 가능:
# from DATA.us_target_ticker_list_screened_20260411 import TARGET_TICKER_LIST

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ── DB 연결 ──────────────────────────────────────────────────────
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print(f"[OK] Import 완료")
print(f"[설정] FORECAST_HORIZON={FORECAST_HORIZON}Q  ERP_METHOD={ERP_METHOD}")
print(f"[설정] WACC 범위 [{WACC_FLOOR:.0%}, {WACC_CAP:.0%}]  Phase2 g 범위 [{G_FLOOR_MIN:.0%}, {G_CEIL:.0%}]")


[01:02:09][DB] 연결 성공 host=192.168.0.230 port=3307
[OK] Import 완료
[설정] FORECAST_HORIZON=8Q  ERP_METHOD=damodaran_floor
[설정] WACC 범위 [5%, 20%]  Phase2 g 범위 [-20%, 100%]


## Cell 4 · 결과 테이블 초기화 (SAVE_TO_DB=True 일 때만)

In [16]:
# ═══════════════════════════════════════════════════════════════
#  결과 테이블 초기화 (SAVE_TO_DB=True 일 때만) — 사용자 입력 없음
# ═══════════════════════════════════════════════════════════════

if not SAVE_TO_DB:
    log("DB", "SAVE_TO_DB=False → 결과 테이블 초기화 생략 (Excel 출력만 수행)")
else:
    # ── 결과 테이블 생성 (없으면) ─────────────────────────────────
    CREATE_SQL = f'''
    CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
      `id`                BIGINT      NOT NULL AUTO_INCREMENT,
      `date`              DATE        NOT NULL COMMENT '평가 실행일',
      `ticker`            VARCHAR(20) NOT NULL COMMENT 'A005930 형식',
      `quarter`           VARCHAR(10)          COMMENT 'e.g. 2026Q1',
      `sales_actual`      DOUBLE               COMMENT '과거 Sales (원)',
      `sales_forecast`    DOUBLE               COMMENT '예측 Sales (원)',
      `opm_forecast`      DOUBLE,
      `ebit`              DOUBLE,
      `tax_rate`          DOUBLE,
      `nopat`             DOUBLE,
      `da`                DOUBLE,
      `capex`             DOUBLE,
      `nwc`               DOUBLE,
      `delta_nwc`         DOUBLE,
      `fcff`              DOUBLE,
      `roic`              DOUBLE,
      `reinvestment_rate` DOUBLE,
      `g_terminal`        DOUBLE,
      `discount_rate`     DOUBLE COMMENT 'WACC',
      `wacc_re`           DOUBLE COMMENT 'Cost of Equity',
      `wacc_rd`           DOUBLE COMMENT 'Cost of Debt',
      `beta_raw`          DOUBLE,
      `beta_blume`        DOUBLE,
      `enterprise_value`  DOUBLE,
      `net_debt`          DOUBLE,
      `equity_value`      DOUBLE,
      `shares`            DOUBLE,
      `target_price`      DOUBLE,
      `current_price`     DOUBLE,
      `upside_pct`        DOUBLE,
      `moat_label`        VARCHAR(30),
      `eva_spread`        DOUBLE,
      `moat_rho`          DOUBLE      COMMENT '★v8 Phase2 지속성 ρ (적응형 상한 반영)',
      `rho_cap_applied`   TINYINT     COMMENT '★v8 g0 기반 ρ 적응형 상한 적용 여부',
      `tv_weight_pct`     DOUBLE      COMMENT '★v8 TV 가 EV 에서 차지하는 비중(%)',
      `tv_fcff_mult`      DOUBLE      COMMENT '★v8 TV / 마지막 연간 FCFF 배수',
      `tv_capped`         TINYINT     COMMENT '★v8 TV 배수 상한 적용 여부',
      `mc_ratio`          DOUBLE      COMMENT '★v8 지분가치 / 실제 시총 배율 (Sanity)',
      `revenue_quarters`  INT COMMENT '사용된 매출 분기 수',
      `forecast_model`    VARCHAR(20) COMMENT 'Sales 예측 모델명',
      `forecast_date`     DATE COMMENT '매출 예측 실행일',
      `nwc_method`        VARCHAR(20) COMMENT 'operating_v7 | legacy_proxy',
      `nwc_to_sales`      DOUBLE      COMMENT 'gamma = NWC/Sales 계수',
      `op_current_assets` DOUBLE      COMMENT '비현금 영업유동자산(최근, 원)',
      `op_current_liab`   DOUBLE      COMMENT '비차입 영업유동부채(최근, 원)',
      `created_at`        DATETIME    DEFAULT CURRENT_TIMESTAMP,
      PRIMARY KEY (`id`),
      UNIQUE KEY uq_main (`ticker`, `date`, `quarter`),
      INDEX idx_ticker (`ticker`),
      INDEX idx_date   (`date`)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    '''

    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(CREATE_SQL)
        conn.commit()
        log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
    finally:
        conn.close()

    # ★ v7: 기존 테이블에 신규 컬럼 보강 (이미 있으면 무시)
    _ALTER_COLS = [
        ("nwc_method",        "VARCHAR(20)"),
        ("nwc_to_sales",      "DOUBLE"),
        ("op_current_assets", "DOUBLE"),
        ("op_current_liab",   "DOUBLE"),
        ("moat_rho",          "DOUBLE"),     # ★ v8
        ("rho_cap_applied",   "TINYINT"),    # ★ v8
        ("tv_weight_pct",     "DOUBLE"),     # ★ v8
        ("tv_fcff_mult",      "DOUBLE"),     # ★ v8
        ("tv_capped",         "TINYINT"),    # ★ v8
        ("mc_ratio",          "DOUBLE"),     # ★ v8
    ]
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            for _c, _t in _ALTER_COLS:
                try:
                    cur.execute(f"ALTER TABLE `{TABLE_RESULT}` ADD COLUMN `{_c}` {_t}")
                except Exception:
                    pass  # 이미 존재
        conn.commit()
    finally:
        conn.close()


[01:02:10][DB] SAVE_TO_DB=False → 결과 테이블 초기화 생략 (Excel 출력만 수행)


## Cell 5 · 시장 파라미터 (Rf, KOSPI, E(Rm)) — 1회 계산 후 캐싱

모든 종목이 공유하는 시장 공통 파라미터:
- `RF` : BOK API → 10Y 국고채 (실시간)
- `KOSPI_PX` : KOSPI 종합지수 (베타·E(Rm) 계산용)
- `E_RM`, `ERP` : Damodaran 또는 KOSPI geo 10y

In [17]:
# 1. Risk-free rate (BOK API)
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
log("MKT", f"Rf = {RF:.4%}  (source: {RF_SOURCE})")

# 2. KOSPI 시계열 (12년치 → 베타·E(Rm) 모두 사용)
KOSPI_PX = load_kospi_series(start_date=(datetime.today() - timedelta(days=365*12)).strftime("%Y-%m-%d"))
log("MKT", f"KOSPI {len(KOSPI_PX):,}거래일  ({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")
log("MKT", f"  현재 KOSPI = {KOSPI_PX.iloc[-1]:,.2f}")

# 3. E(Rm) 추정
mkt = estimate_market_return(
    method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX,
    years=10, damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR,
)
E_RM = mkt["e_rm"]
ERP  = mkt["erp"]
log("MKT", f"E(Rm) = {E_RM:.4%}  ERP = {ERP:.4%}")
log("MKT", f"  → {mkt['note']}")
if mkt["kospi_geo"] is not None:
    log("MKT", f"  (참고) KOSPI 10년 기하평균 = {mkt['kospi_geo']:.4%}")

print()
print("=" * 60)
print(f"  Rf  = {RF:>7.3%}    (BOK 10Y 국고채)")
print(f"  ERP = {ERP:>7.3%}    (방식: {ERP_METHOD})")
print(f"  E(Rm) = {E_RM:>7.3%}")
print("=" * 60)


[01:02:10][MKT] Rf = 4.5390%  (source: BOK_2026-09-28)
[KOSPI] try source=fdr (2014-10-02 ~ 2026-09-29) ... FAIL (FDR 실패 (재시도 3회): LOGOUT)
[KOSPI] try source=pykrx (2014-10-02 ~ 2026-09-29) ... FAIL ('지수명')
[KOSPI] try source=yfinance (2014-10-02 ~ 2026-09-29) ... OK  2,936거래일
[01:02:18][MKT] KOSPI 2,936거래일  (2014-10-02 ~ 2026-09-23)
[01:02:18][MKT]   현재 KOSPI = 7,080.92
[01:02:18][MKT] E(Rm) = 11.5390%  ERP = 7.0000%
[01:02:18][MKT]   → Damodaran 한국 ERP 7.0% 적용
[01:02:18][MKT]   (참고) KOSPI 10년 기하평균 = 13.1761%

  Rf  =  4.539%    (BOK 10Y 국고채)
  ERP =  7.000%    (방식: damodaran_floor)
  E(Rm) = 11.539%


## Cell 6 · `load_korea_revenue_forecast` 패치 (+ v1.4 ticker 형식 혼재 흡수)

구스키마 created_at 버그(매출 forecast 상수 복제) 수정판. updated_at 버전 선택 + 3중 검증(분기 수·연속성·상수 시계열).

> ⚠️ 만약 Cell 3(import 셀)을 다시 실행하면 패치가 풀리므로, 이 셀도 반드시 재실행하세요.

In [18]:
# ═══════════════════════════════════════════════════════════════
#  PATCH 셀 · load_korea_revenue_forecast 교체
#  (구스키마 created_at 버그 → 매출 forecast 상수 복제 문제 수정)
#
#  ▸ 붙여넣을 위치 : korea_fcff_dcf_valuation_v7.ipynb 의
#                    import 셀(DATA.korea_valuation_helpers import) "바로 다음".
#                    KoreaDCFModel 실행(Cell 9/10) 전이면 어디든 OK.
#  ▸ 원리          : 노트북 전역의 load_korea_revenue_forecast 이름을
#                    아래 수정판으로 재바인딩 → KoreaDCFModel.load_sales 가
#                    호출 시점에 이 패치판을 사용.
#  ▸ 주의          : import 셀을 다시 실행하면 패치가 풀리므로,
#                    import 셀 재실행 후에는 이 셀도 재실행할 것.
#
#  ▸ 수정 내용 (korea_revenue_analysis_notebook_v3 의 fetch_forecast 와 동일 원리)
#    1) 버전 식별: MAX(created_at) → MAX(updated_at) (자동 감지, 없으면 created_at)
#       - 구스키마 UNIQUE KEY(date,ticker,indicator) 에서는 재예측 시
#         겹치는 분기의 created_at 이 갱신되지 않아 '새로 생긴 분기 1개'만
#         최신 버전으로 잡히는 문제가 있었음 (예: A031330 → 2028Q1 한 분기).
#    2) 같은 분기 중복 시 가장 최근 갱신분만 사용.
#    3) 검증 추가 — 아래 셋 중 하나라도 걸리면 ValueError 로 즉시 차단
#       (잘못된 valuation 이 DB 에 저장되는 것을 막기 위함):
#       a. 분기 수 < horizon
#       b. 분기 불연속 (중간 분기 누락)
#       c. 전 분기 동일값 (상수 시계열 — 이번 버그의 증상)
# ═══════════════════════════════════════════════════════════════
from sqlalchemy import text as _sa_text

_FC_MODEL_PRIORITY = ("Ensemble", "SARIMA", "ETS", "Theta")
_FC_RECENCY_CACHE: dict = {}


def _fc_recency_col(table_name: str) -> str:
    """updated_at 컬럼이 있으면 그것을, 없으면 created_at 을 버전 기준으로 사용."""
    if table_name in _FC_RECENCY_CACHE:
        return _FC_RECENCY_CACHE[table_name]
    sql = ("SELECT COLUMN_NAME FROM information_schema.COLUMNS "
           "WHERE TABLE_SCHEMA = :db AND TABLE_NAME = :tbl")
    with engine.connect() as conn:
        cols = {r[0].lower() for r in conn.execute(
            _sa_text(sql), {"db": db_info["database"], "tbl": table_name}
        ).fetchall()}
    rc = "updated_at" if "updated_at" in cols else "created_at"
    _FC_RECENCY_CACHE[table_name] = rc
    if rc == "created_at":
        log("PATCH", f"[주의] {table_name} 에 updated_at 없음 → created_at 기준. "
                     "구스키마면 일부 종목 분기 누락 가능 — updated_at 추가 권장.")
    return rc


def load_korea_revenue_forecast(ticker: str,
                                db_info: dict,
                                table_name: str = "korea_revenue_forecast_result",
                                horizon: int = 8):
    """[PATCHED v7-fix] 최신 '실행 버전' 전체 분기를 안전하게 로드.

    Returns
    -------
    (pd.Series, str, str)
        forecast : 천원 단위, index=분기 date (호출측에서 ×1000 변환)
        model    : 사용된 indicator (Ensemble 우선)
        run_date : 예측 실행 버전 날짜 (YYYY-MM-DD)
    """
    rc = _fc_recency_col(table_name)

    # ★ v1.4: 결과 테이블 ticker 형식 혼재 흡수
    #   Korea_revenue_forecast v5+ 는 RESULT_TICKER_HAS_A_PREFIX=False 로
    #   '003350' 형식으로 저장하는 반면, 과거 버전/타 파이프라인은 'A003350'
    #   형식으로 저장 → 한 형식만 조회하면 최신 재예측을 놓치고 구버전
    #   (stale) 예측을 로드함 (A003350 사례). 두 형식을 함께 조회하고,
    #   최신 실행 버전 선택은 기존 MAX(DATE) 로직이 형식 무관하게 처리.
    _t = str(ticker).strip()
    _body = _t[1:] if _t.upper().startswith("A") else _t
    t_a, t_no = f"A{_body}", _body

    df, used_model, run_date = None, "", None
    for model in _FC_MODEL_PRIORITY:
        # indicator 가 'Ensemble' 형태든 '매출액(천원)_Ensemble' 형태든 모두 매칭
        with engine.connect() as conn:
            row = conn.execute(_sa_text(
                f"SELECT MAX(DATE({rc})) FROM {table_name} "
                f"WHERE ticker IN (:t, :t2) AND (indicator = :m OR indicator LIKE :ml)"
            ), {"t": t_a, "t2": t_no, "m": model, "ml": f"%\\_{model}"}).fetchone()
        if row is None or row[0] is None:
            continue
        run_date = str(row[0])
        cand = pd.read_sql(_sa_text(
            f"SELECT date, value, {rc} AS run_ts FROM {table_name} "
            f"WHERE ticker IN (:t, :t2) AND (indicator = :m OR indicator LIKE :ml) "
            f"  AND DATE({rc}) = :fd "
            f"ORDER BY date"
        ), engine, params={"t": t_a, "t2": t_no, "m": model,
                           "ml": f"%\\_{model}", "fd": run_date})
        if not cand.empty:
            df, used_model = cand, model
            break

    if df is None:
        return pd.Series(dtype=float), "", None

    df["date"]   = pd.to_datetime(df["date"])
    df["run_ts"] = pd.to_datetime(df["run_ts"])
    df = (df.sort_values(["date", "run_ts"])
            .drop_duplicates(subset=["date"], keep="last"))
    fc = df.set_index("date")["value"].astype(float).sort_index()

    # ── 검증 a: 분기 수 ───────────────────────────────────────
    if len(fc) < horizon:
        raise ValueError(
            f"[{ticker}] forecast 분기 {len(fc)}개 < horizon {horizon} "
            f"(version={rc} {run_date}, indicator={used_model}). "
            f"구스키마 created_at 잔존 가능성 — 해당 종목 재예측 또는 "
            f"테이블 updated_at 마이그레이션 필요."
        )
    fc = fc.iloc[:horizon]

    # ── 검증 b: 분기 연속성 ──────────────────────────────────
    per = fc.index.to_period("Q")
    gaps = (per[1:].astype("int64") - per[:-1].astype("int64"))
    if (gaps != 1).any():
        bad = [str(per[i + 1]) for i, g in enumerate(gaps) if g != 1]
        raise ValueError(
            f"[{ticker}] forecast 분기 불연속: {bad} "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    # ── 검증 c: 상수 시계열 (이번 버그의 증상) ────────────────
    if fc.nunique() == 1:
        raise ValueError(
            f"[{ticker}] forecast {horizon}개 분기가 모두 동일값"
            f"({fc.iloc[0]:,.0f}천원) — 단일 분기 복제 의심. "
            f"(version={rc} {run_date}). 해당 종목 재예측 필요."
        )

    return fc, used_model, run_date


# log("PATCH", "load_korea_revenue_forecast → updated_at 버전 선택 + 3중 검증 패치 적용 완료")

## Cell 7 · KoreaDCFModel v8 클래스 (★ US v13 Validity 가드 이식 + v1.3 forecast 정렬 가드)

v8 원본과 동일 — Rd≥Rf · WACC≥max(5%,Rf+1%) · g_term≤min(GDP,Rf) · TV 스프레드 2%+배수 35배 · Sanity Guard(±30배) · g0 기반 ρ 적응형 상한

**v1.5**: `estimate_opm` 회귀 기반 개편(고정 50% 상한 폐지) + `_apply_midcycle` (Phase 2 미드사이클 OPM 수렴)

In [19]:
# ═══════════════════════════════════════════════════════════════
#  KoreaDCFModel v8 (+ v1.5 OPM 회귀 · 미드사이클)  —  한국 주식 Sales-driven FCFF DCF
#  (US v11 v8 + v9.2 패치 + ★ US v13 Validity 가드 이식)
#  ─────────────────────────────────────────────────────────────────
#  v7 → v8 변경 (US v13 Validity Patch 이식):
#    ② compute_cost_of_debt : Rd ≥ Rf 하한
#       compute_wacc        : WACC 하한 = max(5%, Rf+1%)  (구 절대 5%)
#    ③ compute_terminal_growth : g_term ≤ min(GDP, Rf)
#       compute_valuation      : 최소 스프레드 2%(구 1%) + TV/FCFF 배수 ≤ 35배
#    ④ compute_valuation       : Sanity Guard (지분가치/시총 ±30배)
#    ⑤ _estimate_phase2_growth : g0 기반 ρ 적응형 상한
#  v1.4 → v1.5 (노트북 버전):
#    ⑥ estimate_opm : 매출-영업이익 회귀(level/diff/log, 홀드아웃 자동선택) 기본,
#                     고정 OPM 상한 50% 폐지 → [창 내 최저−10%p, 최고+10%p]
#    ⑦ compute_valuation : Phase 2 미드사이클 OPM 수렴 (FCFF 에 마진 차이 × 매출 × (1−T) 반영)
# ═══════════════════════════════════════════════════════════════

class KoreaDCFModel:
    """
    한국 기업 FCFF DCF Valuation Model — v8 (US v13 Validity 가드 이식).

    사용 흐름:
        m = KoreaDCFModel(ticker="A005930", engine=engine, db_info=db_info,
                          rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX, verbose=True)
        m.run()
        m.save_to_db("2026-04-21")
        m.plot()

    v1 → v2 변경사항:
      [v8 패치]
        - NWC_FIELD_ALIASES 매핑 + _compute_bs_nwc_series()
        - estimate_nwc_coef() 가 _last_actual_nwc 저장
        - compute_fcff() prev_nwc 를 γ × last_actual_sales 로 시드
        - _compute_historical_fcff() 첫 분기 prev_nwc 를 γ × first_sales 로 시드
      [v9.2 패치]
        - _estimate_phase2_growth(): 다중 fallback 체인
          · Method 1: 8Q-CAGR (양수 케이스만)
          · Method 2: 12Q-TTM rolling (★ 신규, 부호 반전 명시 처리)
          · Method 3: Forecast yr1/yr2 (★ 신규, g0_fc 로 항상 계산)
        - g0_hist + g0_fc 결합 로직 (min, forecast 우선, 부호 불일치)
        - 음수 g₀ 허용 (G_FLOOR_MIN = -0.20)
        - ρ OLS 보정 (음수 annual 도 회귀 포함)
        - AR(1) clip 하한 G_FLOOR_MIN 으로 완화

    한국 특화 (v1 동일):
      - FMP API 호출 없음 → 모두 DB 조회
      - 자체 베타 계산 (KSE_Price + KOSPI 일별수익률 10년 회귀, Blume)
      - NWC ≈ Receivables + Inventories - Short-term debts  (한국 BS 보완)
      - DataQualityReport 로 모든 fallback 추적
    """

    # ─────────────────────────────────────────────────────────────
    # ★ v8 패치: NWC 필드 alias 매핑 (silent 0 대체 방지)
    #
    # 한국 DataGuide 의 정규화된 컬럼명을 표준 키 → 후보 컬럼들로 매핑.
    # 핵심 컬럼이 없거나 전부 0 이면 verbose 모드에서 경고 출력.
    # ─────────────────────────────────────────────────────────────
    NWC_FIELD_ALIASES = {
        # 자산 측
        "receivables":      ["receivables", "trade_receivables",
                             "accounts_receivable"],
        "inventories":      ["inventories", "inventory"],
        # 부채 측 (단기성)
        "short_term_debt":  ["short_term_debt", "short_term_borrowings"],
        "current_lt_debt":  ["current_lt_debt", "current_portion_of_lt_debt"],
        "lease_liab":       ["lease_liab", "lease_liabilities", "lease_liability"],
    }

    # ─────────────────────────────────────────────────────────────
    # ★ v7: 영업운전자본 alias (재무부채 제외 — 순수 영업 항목만)
    #   자산 측: 매출채권 + 재고자산 + 선급비용
    #   부채 측: 매입채무 + 미지급비용 + 미지급금 + 선수금 + 계약부채
    #   각 std_key 의 첫 후보가 helper(DG_ITEM_CODES) 권장 키와 일치.
    # ─────────────────────────────────────────────────────────────
    NWC_OP_ASSET_ALIASES = {
        "receivables":       ["receivables", "trade_receivables", "accounts_receivable"],
        "inventories":       ["inventories", "inventory"],
        "prepaid_expenses":  ["prepaid_expenses", "prepaid", "prepaid_exp"],
    }
    NWC_OP_LIAB_ALIASES = {
        "payables":             ["payables", "trade_payables", "accounts_payable"],
        "accrued_expenses":     ["accrued_expenses", "accrued_exp", "accrued"],
        "other_payables":       ["other_payables", "non_trade_payables", "other_payable"],
        "advances_received":    ["advances_received", "advance_received", "customer_advances", "unearned"],
        "contract_liabilities": ["contract_liabilities", "contract_liab", "deferred_revenue"],
    }

    def __init__(self, ticker, engine, db_info,
                 rf, e_rm, kospi_series,
                 forecast_horizon=FORECAST_HORIZON,
                 min_history=MIN_HISTORY,
                 gdp_growth=GDP_GROWTH,
                 verbose=False):
        self.ticker_dg     = to_dg_ticker(ticker)
        self.ticker_price  = to_price_ticker(ticker)
        self.engine        = engine
        self.db_info       = db_info
        self.rf            = rf
        self.e_rm          = e_rm
        self.erp           = e_rm - rf
        self.kospi         = kospi_series
        self.horizon       = forecast_horizon
        self.min_history   = min_history
        self.gdp_growth    = gdp_growth
        self.verbose       = verbose

        # 데이터 저장소
        self._sales_actual:   Optional[pd.Series]    = None
        self._sales_forecast: Optional[pd.Series]    = None
        self._used_model:     str                    = ""
        self._forecast_date:  Optional[date]         = None
        self._fs_wide:        Optional[pd.DataFrame] = None
        self._stock_data:     Optional[pd.DataFrame] = None
        self._fcff_history:   Optional[pd.Series]    = None
        self._beta_info:      Optional[Dict]         = None
        self._mkt_cap:        Optional[float]        = None
        self._current_price:  Optional[float]        = None
        self._last_actual_nwc: Optional[float]       = None  # ★ v8
        self._nwc_method:      str                    = "?"      # ★ v7
        self._op_ca_latest:    float                  = np.nan   # ★ v7
        self._op_cl_latest:    float                  = np.nan   # ★ v7
        self._nwc_to_sales:    float                  = np.nan   # ★ v7
        self._opm_info:        Dict                   = {}       # ★ v1.5 OPM 추정 진단

        self.result_df: Optional[pd.DataFrame]       = None
        self.valuation: Optional[Dict]               = None
        self.report = DataQualityReport(ticker=self.ticker_dg)

    # ─────────────────────────────────────────────────────────
    # 1. 데이터 로드
    # ─────────────────────────────────────────────────────────

    def load_sales(self):
        """매출 actual + forecast 로드. 핵심항목이라 결측 시 즉시 raise."""
        # Actual
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=["revenue"], fillna_zero=False,
        )
        actual = wide["revenue"].dropna() * FS_UNIT_MULTIPLIER  # 천원 → 원
        if actual.empty:
            self.report.add("revenue_actual", "missing", n_obs=0,
                            note="korea_fs_data_from_DG에 매출 데이터 없음")
            raise ValueError(f"[{self.ticker_dg}] Sales actual 데이터 없음")

        if len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual),
                            note=f"매출 {len(actual)}분기 < 최소 {MIN_REVENUE_QUARTERS}")
            raise ValueError(
                f"[{self.ticker_dg}] 매출 {len(actual)}분기 < 최소 {MIN_REVENUE_QUARTERS}분기"
            )
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        # Forecast
        forecast, model_name, ca = load_korea_revenue_forecast(
            self.ticker_dg, self.db_info, table_name=TABLE_FORECAST,
            horizon=self.horizon,
        )
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note="korea_revenue_forecast_result에 예측 없음")
            raise ValueError(
                f"[{self.ticker_dg}] 매출 forecast 없음 — Korea_revenue_forecast 먼저 실행"
            )
        forecast = forecast * FS_UNIT_MULTIPLIER  # 천원 → 원

        # ─────────────────────────────────────────────────────
        # ★ v1.3: forecast ↔ actual 시점 정렬 가드 (stale forecast)
        #   증상: DB 예측이 구버전(예: 2025Q3 기준 → 2025Q4~ 시작)인데
        #         actual 은 그 뒤(2026Q2)까지 갱신 → 이미 실현된 분기가
        #         "예측"으로 중복 편입 + 첫 분기 ΔNWC 왜곡.
        #   처리: ① actual 마지막 분기 이하의 forecast 분기 제거
        #         ② 전부 겹치면 raise (재예측 필요)
        #         ③ 첫 forecast 분기 ≠ actual+1Q → 불연속 raise
        #         ④ 잔여 < horizon → ALLOW_SHORT_FORECAST 에 따라
        #            축소 진행(경고) 또는 raise
        # ─────────────────────────────────────────────────────
        last_per = actual.index[-1].to_period("Q")
        fc_per   = forecast.index.to_period("Q")
        stale    = fc_per <= last_per
        self._forecast_dropped_stale = []
        if stale.any():
            self._forecast_dropped_stale = [str(p) for p in fc_per[stale]]
            forecast = forecast[~stale]
            msg = (f"stale forecast: actual({last_per}) 이전/동일 분기 "
                   f"{len(self._forecast_dropped_stale)}개 제거 "
                   f"{self._forecast_dropped_stale} (예측버전={ca})")
            self.report.warn(msg)
            if self.verbose:
                log(self.ticker_dg, f"⚠️  {msg}")

        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note=f"forecast 전 분기가 actual({last_per})과 겹침")
            raise ValueError(
                f"[{self.ticker_dg}] forecast 전 분기가 actual({last_per})과 겹침 — "
                f"예측버전({ca})이 오래됨. Korea_revenue_forecast 재실행 필요.")

        first_per = forecast.index[0].to_period("Q")
        if (first_per - last_per).n != 1:
            raise ValueError(
                f"[{self.ticker_dg}] forecast 시작({first_per})이 "
                f"actual 마지막({last_per})+1Q 와 불연속 — "
                f"actual 중간 분기 누락 또는 forecast 결함. 데이터 점검 필요.")

        if len(forecast) < self.horizon:
            short_msg = (f"정렬 후 forecast {len(forecast)}Q < horizon {self.horizon}Q "
                         f"(예측버전={ca} 이 오래됨 — 재예측 시 전체 horizon 복원)")
            if ALLOW_SHORT_FORECAST:
                self.report.warn(short_msg + " → 축소 horizon 으로 진행")
                if self.verbose:
                    log(self.ticker_dg, f"⚠️  {short_msg} → Phase1={len(forecast)}Q 로 진행")
                self.horizon = len(forecast)
            else:
                raise ValueError(f"[{self.ticker_dg}] {short_msg} "
                                 f"(ALLOW_SHORT_FORECAST=False)")

        self.report.add("revenue_forecast", "ok", n_obs=len(forecast),
                        note=f"model={model_name}, created_at={ca}, "
                             f"start={first_per}, stale_dropped="
                             f"{len(self._forecast_dropped_stale)}")

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:self.horizon]
        self._used_model     = model_name
        self._forecast_date  = ca

        if self.verbose:
            log(self.ticker_dg,
                f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q "
                f"model={model_name} created_at={ca}")
        return self

    def load_financials(self):
        """IS / BS / CF wide-form + 주식수 로드 + (★ v8) NWC 핵심 필드 sanity check."""
        # 재무 항목 wide-form
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=None,  # 전체
            fillna_zero=False,
        )
        if wide.empty:
            raise ValueError(f"[{self.ticker_dg}] 재무 데이터 wide-form 비어있음")

        # 단위 변환: 천원 → 원 (주식수만 제외)
        non_share_cols = [c for c in wide.columns
                          if c not in ("shares_treasury_adj", "shares_common")]
        wide[non_share_cols] = wide[non_share_cols] * FS_UNIT_MULTIPLIER

        # 핵심항목 검증 (영업이익은 필수)
        if "operating_income" not in wide.columns or wide["operating_income"].dropna().empty:
            self.report.add("operating_income", "missing", n_obs=0,
                            note="영업이익 데이터 없음 — 가치평가 불가")
            raise ValueError(f"[{self.ticker_dg}] operating_income 데이터 없음")
        self.report.add("operating_income", "ok",
                        n_obs=int(wide["operating_income"].notna().sum()))

        # ★ v8: NWC 핵심 필드 sanity check
        for std_name, aliases in self.NWC_FIELD_ALIASES.items():
            present = [a for a in aliases if a in wide.columns]
            if not present:
                if self.verbose:
                    log(self.ticker_dg,
                        f"⚠️  NWC 필드 없음: {std_name} (aliases={aliases})")
            else:
                col = pd.to_numeric(wide[present[0]], errors="coerce")
                if col.abs().sum() == 0:
                    if self.verbose:
                        log(self.ticker_dg,
                            f"⚠️  NWC 필드가 전부 0: {present[0]} ({std_name})")

        self._fs_wide = wide
        if self.verbose:
            log(self.ticker_dg, f"FS wide  shape={wide.shape}  "
                f"기간={wide.index.min().date()} ~ {wide.index.max().date()}")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. 변수별 추정
    # ─────────────────────────────────────────────────────────

    @staticmethod
    def _winsorize(s, limits=WINSORIZE_LIMITS):
        s = s.dropna()
        if len(s) < 4:
            return s
        lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
        return s.clip(lo, hi)

    @staticmethod
    def _ols_ratio(x, y):
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES:
            return np.nan, -1.0, int(mask.sum())
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return float(slope), float(r ** 2), int(mask.sum())

    def _resolve_alias(self, std_name: str) -> Optional[str]:
        """★ v8: NWC_FIELD_ALIASES 의 std_name 에 대해 wide 컬럼에 존재하는 첫 alias 반환."""
        wide = self._fs_wide
        if wide is None:
            return None
        for alias in self.NWC_FIELD_ALIASES.get(std_name, [std_name]):
            if alias in wide.columns:
                return alias
        return None

    def _resolve_from(self, alias_map: dict, std_name: str) -> Optional[str]:
        """★ v7: 임의 alias_map 에서 wide 컬럼에 존재하는 첫 alias 반환."""
        wide = self._fs_wide
        if wide is None:
            return None
        for alias in alias_map.get(std_name, [std_name]):
            if alias in wide.columns:
                return alias
        return None

    def _sum_components(self, alias_map: dict):
        """★ v7: alias_map 의 각 std_key 를 resolve → 결측 0 으로 합산.
        반환: (합산 Series, 실제 resolve 된 std_key 리스트)."""
        wide = self._fs_wide
        total = pd.Series(0.0, index=wide.index)
        resolved = []
        for std_name in alias_map:
            col = self._resolve_from(alias_map, std_name)
            if col is not None:
                total = total + pd.to_numeric(wide[col], errors="coerce").fillna(0)
                resolved.append(std_name)
        return total, resolved

    def estimate_tax_rate(self):
        """실효세율 (pretaxIncome > 0 분기, clip 0~40%, 한국 법정세율 22% fallback)"""
        df = self._fs_wide[["pretax_income", "tax_expense"]].dropna()
        df = df[df["pretax_income"] > 0]
        if df.empty:
            self.report.add("tax_rate", "fallback_zero", n_obs=0,
                            value=0.22, note="법정세율 22% fallback")
            return 0.22
        rates = (df["tax_expense"] / df["pretax_income"]).clip(0, 0.40)
        med = float(rates.median())
        self.report.add("tax_rate", "ok", n_obs=len(rates), value=med)
        return med

    # ─────────────────────────────────────────────────────────
    # ★ v1.5: OPM 예측 — 매출-영업이익 회귀 (기본) / 시계열 (선택)
    # ─────────────────────────────────────────────────────────
    # ─────────────────────────────────────────────────────────
    # ★ v1.5: OPM 예측 — 매출-영업이익 회귀 (기본) / 시계열 (선택)
    # ─────────────────────────────────────────────────────────
    def _opm_hist_frame(self) -> pd.DataFrame:
        """분기 매출·영업이익 (원). index=분기말, 매출>0 인 분기만."""
        df = self._fs_wide[["operating_income", "revenue"]].apply(
            pd.to_numeric, errors="coerce").dropna()
        df = df[df["revenue"] > 0].copy()
        df.index = pd.to_datetime(df.index)
        df["per"] = df.index.to_period("Q")
        df = df[~df["per"].duplicated(keep="last")]
        df["opm"] = df["operating_income"] / df["revenue"]
        return df

    def _opm_clip_bounds(self, opm_window: pd.Series) -> Tuple[float, float]:
        """예측 OPM 허용 범위 = [창 내 최저 − m, 최고 + m] ∩ OPM_ABS_BOUNDS (고정 50% 상한 폐지)."""
        lo = float(opm_window.min()) - OPM_CLIP_MARGIN
        hi = float(opm_window.max()) + OPM_CLIP_MARGIN
        return max(lo, OPM_ABS_BOUNDS[0]), min(hi, OPM_ABS_BOUNDS[1])

    @staticmethod
    def _qdummies(per_index) -> np.ndarray:
        q = np.array([p.quarter for p in per_index])
        return np.column_stack([(q == k).astype(float) for k in (2, 3, 4)])

    def _fit_op_reg(self, method: str, h: pd.DataFrame) -> Optional[Dict]:
        """영업이익 회귀 적합 (단위: 조원 스케일로 안정화).
          level : OP = a + b·S + 분기더미         (a<0 → 고정비, b → 공헌이익률)
          diff  : ΔOP(YoY) = a + b·ΔS(YoY)        (계절성 제거)
          log   : log OP = a + b·log S            (창 내 전 분기 흑자일 때만)
        """
        U = 1e12
        S, OP = h["revenue"].values / U, h["operating_income"].values / U
        try:
            if method == "level":
                if len(h) < 8:
                    return None
                X = np.column_stack([np.ones(len(S)), S, self._qdummies(h["per"])])
                coef, *_ = np.linalg.lstsq(X, OP, rcond=None)
                fit = X @ coef
                ss_tot = np.sum((OP - OP.mean()) ** 2)
                r2 = 1 - np.sum((OP - fit) ** 2) / ss_tot if ss_tot > 0 else np.nan
                return {"method": "level", "coef": coef, "alpha": coef[0] * U, "beta": coef[1],
                        "r2": float(r2), "n": len(h)}
            if method == "diff":
                s = pd.Series(S, index=h["per"]); o = pd.Series(OP, index=h["per"])
                s4 = s.copy(); s4.index = s4.index + 4        # t 시점에 t-4 값 정렬
                o4 = o.copy(); o4.index = o4.index + 4
                ds = (s - s4).dropna()
                do = (o - o4).dropna()
                d = pd.concat([ds.rename("ds"), do.rename("do")], axis=1).dropna()
                if len(d) < 8:
                    return None
                b, a, r, _, _ = stats.linregress(d["ds"].values, d["do"].values)
                return {"method": "diff", "coef": np.array([a, b]), "alpha": a * U, "beta": b,
                        "r2": float(r ** 2), "n": len(d)}
            if method == "log":
                if (OP <= 0).any() or len(h) < 8:
                    return None
                b, a, r, _, _ = stats.linregress(np.log(S), np.log(OP))
                resid = np.log(OP) - (a + b * np.log(S))
                smear = float(np.mean(np.exp(resid)))          # Duan smearing
                return {"method": "log", "coef": np.array([a, b]), "alpha": a, "beta": b,
                        "r2": float(r ** 2), "n": len(h), "smear": smear}
        except Exception:
            return None
        return None

    def _predict_op_reg(self, fit: Dict, h: pd.DataFrame, sales_future: pd.Series) -> pd.Series:
        """sales_future(원, index=분기말) → 예측 영업이익(원). diff 는 t-4 체인 (실적 우선)."""
        U = 1e12
        per_f = pd.to_datetime(sales_future.index).to_period("Q")
        S_f = sales_future.values / U
        m = fit["method"]
        if m == "level":
            X = np.column_stack([np.ones(len(S_f)), S_f, self._qdummies(per_f)])
            op = X @ fit["coef"]
        elif m == "log":
            op = np.exp(fit["coef"][0] + fit["coef"][1] * np.log(np.maximum(S_f, 1e-9))) * fit["smear"]
        else:  # diff
            s_all = pd.Series(h["revenue"].values / U, index=h["per"])
            o_all = pd.Series(h["operating_income"].values / U, index=h["per"])
            a, b = fit["coef"]
            op = []
            for p, s in zip(per_f, S_f):
                p4 = p - 4
                if p4 in o_all.index and p4 in s_all.index:
                    v = o_all[p4] + a + b * (s - s_all[p4])
                else:
                    v = np.nan
                op.append(v)
                s_all.loc[p], o_all.loc[p] = s, v
            op = np.array(op, dtype=float)
        return pd.Series(op * U, index=sales_future.index)

    def _opm_regression(self, sales_series: pd.Series) -> Optional[pd.Series]:
        """회귀 기반 OPM 경로. 실패 시 None (→ 호출부에서 ts fallback)."""
        hist = self._opm_hist_frame()
        h = hist.tail(OPM_REG_WINDOW_Q)
        info = {"mode_used": "regression", "window_q": len(h),
                "window": f"{h['per'].iloc[0]}~{h['per'].iloc[-1]}" if len(h) else None}
        if len(h) < self.min_history:
            info["fallback_reason"] = f"회귀 표본 {len(h)} < MIN_HISTORY {self.min_history}"
            self._opm_info.update(info)
            return None

        cands = ["level", "diff", "log"] if OPM_REG_METHOD == "auto" else [OPM_REG_METHOD]
        H = OPM_HOLDOUT_Q
        mae = {}
        for mth in cands:                       # 최근 H분기 홀드아웃 (실제 매출로 OP 재현)
            tr, te = h.iloc[:-H], h.iloc[-H:]
            f = self._fit_op_reg(mth, tr)
            if f is None:
                continue
            pred = self._predict_op_reg(f, tr, pd.Series(te["revenue"].values, index=te.index))
            err = np.abs(pred.values - te["operating_income"].values)
            if np.isfinite(err).all():
                mae[mth] = float(err.mean())
        order = sorted(mae, key=mae.get) if mae else cands
        fit = None
        for mth in order:
            fit = self._fit_op_reg(mth, h)
            if fit is not None:
                break
        if fit is None:
            info["fallback_reason"] = "회귀 적합 실패 (전 방식)"
            self._opm_info.update(info)
            return None
        if OPM_MODE == "auto" and (not np.isfinite(fit["r2"]) or fit["r2"] < OPM_REG_MIN_R2):
            info["fallback_reason"] = f"auto: R²={fit['r2']:.2f} < {OPM_REG_MIN_R2}"
            info.update({"reg_method": fit["method"], "reg_r2": fit["r2"]})
            self._opm_info.update(info)
            return None

        op = self._predict_op_reg(fit, h, sales_series)
        opm_raw = (op / sales_series).astype(float)
        if opm_raw.isna().any():                # diff 체인 결손 → level 로 보완
            f_lv = self._fit_op_reg("level", h)
            if f_lv is not None:
                alt = self._predict_op_reg(f_lv, h, sales_series) / sales_series
                opm_raw = opm_raw.fillna(alt)
        lo, hi = self._opm_clip_bounds(h["opm"])
        opm = opm_raw.clip(lo, hi)
        info.update({
            "reg_method": fit["method"], "reg_alpha": fit["alpha"], "reg_beta": fit["beta"],
            "reg_r2": fit["r2"], "reg_n": fit["n"], "holdout_mae": mae,
            "clip_lo": lo, "clip_hi": hi, "n_clipped": int((opm_raw != opm).sum()),
            "opm_raw": opm_raw.tolist(),
        })
        self._opm_info.update(info)
        self.report.add("opm_forecast", "ok", n_obs=fit["n"],
                        note=f"regression[{fit['method']}] R²={fit['r2']:.2f} "
                             f"clip=[{lo:.1%},{hi:.1%}] clipped={info['n_clipped']}")
        if self.verbose:
            a_str = (f"α={fit['alpha']/1e12:,.2f}조 β={fit['beta']:.3f}" if fit["method"] != "log"
                     else f"a={fit['alpha']:.3f} b={fit['beta']:.3f}")
            log(self.ticker_dg, f"[OPM] 회귀[{fit['method']}] {a_str} R²={fit['r2']:.2f} "
                f"n={fit['n']} 홀드아웃MAE(조)={ {k: round(v/1e12, 3) for k, v in mae.items()} } "
                f"→ OPM {opm.iloc[0]:.1%} … {opm.iloc[-1]:.1%} (clip [{lo:.1%},{hi:.1%}])")
        return pd.Series(opm.values, index=sales_series.index)

    def _opm_timeseries(self, sales_series: pd.Series, record: bool = True) -> pd.Series:
        """(구 v1.4 방식) OPM 시계열 SARIMA/ETS/Theta 평균. 고정 50% 대신 동적 clip."""
        hist = self._opm_hist_frame()
        opm_series = self._winsorize(hist["opm"])
        lo, hi = self._opm_clip_bounds(hist["opm"].tail(OPM_REG_WINDOW_Q))

        def _ret(vals, status, note):
            if record:
                self._opm_info.update({"ts_status": status, "ts_note": note,
                                       "clip_lo": lo, "clip_hi": hi})
                self.report.add("opm_forecast", status, n_obs=len(opm_series), note=note)
            return pd.Series(vals, index=sales_series.index)

        if len(opm_series) < self.min_history:
            fb = float(np.clip(opm_series.median() if not opm_series.empty else 0.05, lo, hi))
            return _ret([fb] * len(sales_series), "fallback_median",
                        f"OPM history {len(opm_series)} < {self.min_history}")
        try:
            m = seasonal_periods_from_freq(infer_freq_alias(opm_series.index))
        except Exception:
            m = 4
        forecasts = {}
        for name, fn in [
            ("SARIMA", lambda y: forecast_sarima(y, len(sales_series), seasonal_period=m)),
            ("ETS",    lambda y: forecast_ets(y, len(sales_series), m=m)),
            ("Theta",  lambda y: forecast_theta(y, len(sales_series), m=m)),
        ]:
            try:
                res = fn(opm_series)
                if "forecast" in res and "error" not in res:
                    forecasts[name] = np.asarray(res["forecast"])
            except Exception:
                pass
        if forecasts:
            ens = np.clip(np.mean(list(forecasts.values()), axis=0), lo, hi)
            return _ret(ens, "ok", f"ts models={list(forecasts.keys())} clip=[{lo:.1%},{hi:.1%}]")
        fb = float(np.clip(opm_series.tail(8).median() if len(opm_series) >= 8
                           else opm_series.median(), lo, hi))
        return _ret([fb] * len(sales_series), "fallback_median", "모든 ts 모델 실패")

    def estimate_opm(self, sales_series):
        """★ v1.5 OPM 예측 진입점.
        OPM_MODE = "regression": 매출-영업이익 회귀 (실패 시에만 ts)
                   "auto"      : 회귀 R² ≥ OPM_REG_MIN_R2 이면 회귀, 아니면 ts
                   "ts"        : 구 v1.4 시계열 방식 (동적 clip)
        """
        self._opm_info = {"mode_requested": OPM_MODE}
        if OPM_MODE in ("regression", "auto"):
            r = self._opm_regression(sales_series)
            if r is not None:
                return r
            if self.verbose:
                log(self.ticker_dg, f"[OPM] 회귀 미사용 → ts fallback "
                    f"({self._opm_info.get('fallback_reason')})")
        self._opm_info["mode_used"] = "ts"
        return self._opm_timeseries(sales_series)

    def estimate_ratio_coef(self, target_keys, ratio_name, take_abs=False):
        """
        D&A / CapEx 비율 계수 추정.
        target_keys : 합산할 컬럼 리스트 (예: D&A = ['da_cf', 'intangible_amort_cf'])
        OLS slope 우선 → R²<0.30 or 부족 → median ratio fallback
        """
        # Sales
        sales = self._fs_wide["revenue"].dropna()
        sales = sales[sales > 0]
        if sales.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0,
                            note="revenue 없음")
            return 0.0, "missing"

        # Target value (여러 컬럼 합산, 결측은 0으로)
        target = pd.Series(0.0, index=self._fs_wide.index)
        valid_cols = []
        for k in target_keys:
            if k in self._fs_wide.columns:
                col = self._fs_wide[k].fillna(0.0)
                target = target + col
                valid_cols.append(k)
        if not valid_cols:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0,
                            note=f"{target_keys} 모두 컬럼 없음")
            return 0.0, "missing"
        if take_abs:
            target = target.abs()

        merged = pd.concat([sales.rename("rev"), target.rename("y")],
                           axis=1, join="inner").dropna()
        merged = merged[merged["rev"] > 0]
        if merged.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0,
                            note="merged 비어있음")
            return 0.0, "empty"

        slope, r2, n = self._ols_ratio(merged["rev"], merged["y"])
        use_ols = (not np.isnan(slope) and r2 >= OLS_MIN_R2 and slope >= 0
                   and n >= OLS_MIN_SAMPLES)

        if use_ols:
            self.report.add(ratio_name, "ok", n_obs=n, value=slope, r2=r2,
                            note=f"OLS cols={valid_cols}")
            return float(slope), "ols"
        # median fallback
        ratios = (merged["y"] / merged["rev"]).replace([np.inf, -np.inf], np.nan).dropna()
        if ratios.empty:
            self.report.add(ratio_name, "fallback_zero", n_obs=0, value=0.0)
            return 0.0, "ratio_empty"
        ratios = self._winsorize(ratios)
        med = float(ratios.median())
        self.report.add(ratio_name, "fallback_median", n_obs=len(ratios),
                        value=med, r2=r2,
                        note=f"OLS R²={r2:.2f} → median, cols={valid_cols}")
        return max(med, 0.0), "median_ratio"

    # ─────────────────────────────────────────────────────────
    # ★ v8 패치: NWC 시계열 계산 (alias 매핑 + 표준 산식)
    # ─────────────────────────────────────────────────────────
    def _compute_operating_nwc_series(self):
        """
        ★ v7: 비현금 영업운전자본 시계열.
            NWC = (매출채권 + 재고자산 + 선급비용)
                  − (매입채무 + 미지급비용 + 미지급금 + 선수금 + 계약부채)
        재무부채(단기차입금/유동성장기부채/리스)·현금은 제외 — 순수 영업 운전자본.
        반환: (DataFrame[date, nwc, op_ca, op_cl], 자산 resolve, 부채 resolve, 부채존재여부)
        """
        wide = self._fs_wide
        op_ca, ca_res = self._sum_components(self.NWC_OP_ASSET_ALIASES)
        op_cl, cl_res = self._sum_components(self.NWC_OP_LIAB_ALIASES)
        has_op_liab = len(cl_res) > 0     # 신규 영업부채 항목이 하나라도 있으면 v7 적용
        nwc = op_ca - op_cl
        self._op_ca_latest = float(op_ca.iloc[-1]) if len(op_ca) else np.nan
        self._op_cl_latest = float(op_cl.iloc[-1]) if len(op_cl) else np.nan
        df = pd.DataFrame({"date": wide.index, "nwc": nwc.values}).reset_index(drop=True)
        return df, ca_res, cl_res, has_op_liab

    def _compute_legacy_nwc_series(self) -> pd.DataFrame:
        """
        (구 v6 proxy) NWC = (Recv+Inv) − (단기차입금+유동성장기부채+리스).
        ※ 재무부채 혼입 + 영업매입채무 누락 → 이론적으로 부정확. 폴백/비교용으로만 유지.
        """
        wide = self._fs_wide
        recv_col = self._resolve_alias("receivables")
        inv_col  = self._resolve_alias("inventories")
        rec = pd.to_numeric(wide[recv_col], errors="coerce").fillna(0) if recv_col else pd.Series(0.0, index=wide.index)
        inv = pd.to_numeric(wide[inv_col],  errors="coerce").fillna(0) if inv_col  else pd.Series(0.0, index=wide.index)
        st = pd.Series(0.0, index=wide.index)
        for std_name in ["short_term_debt", "current_lt_debt", "lease_liab"]:
            col = self._resolve_alias(std_name)
            if col:
                st = st + pd.to_numeric(wide[col], errors="coerce").fillna(0)
        nwc = (rec + inv) - st
        self._op_ca_latest = float((rec + inv).iloc[-1]) if len(wide) else np.nan
        self._op_cl_latest = float(st.iloc[-1]) if len(wide) else np.nan
        return pd.DataFrame({"date": wide.index, "nwc": nwc.values}).reset_index(drop=True)

    def _compute_bs_nwc_series(self) -> pd.DataFrame:
        """
        NWC 시계열 디스패처 (★ v7).
          USE_OPERATING_NWC=True 이고 신규 영업부채 항목이 DB 에 적재돼 있으면
          → 영업운전자본(operating_v7). 미적재면 → legacy proxy 로 자동 폴백.
        """
        if USE_OPERATING_NWC:
            df, ca_res, cl_res, has_op_liab = self._compute_operating_nwc_series()
            if has_op_liab:
                self._nwc_method = "operating_v7"
                self.report.add("nwc_definition", "ok", n_obs=len(df),
                                note=f"operating_v7 CA={ca_res} CL={cl_res}")
                if self.verbose:
                    log(self.ticker_dg,
                        f"NWC[v7 영업] CA={ca_res} CL={cl_res}  "
                        f"최근 CA={self._op_ca_latest/1e9:,.1f}B CL={self._op_cl_latest/1e9:,.1f}B "
                        f"NWC={(self._op_ca_latest-self._op_cl_latest)/1e9:,.1f}B")
                return df
            self.report.warn("v7 영업부채 항목 없음(DB 미적재?) → legacy NWC proxy 폴백")
            if self.verbose:
                log(self.ticker_dg, "⚠️  v7 영업부채 항목 없음 → legacy proxy 사용")
        self._nwc_method = "legacy_proxy"
        return self._compute_legacy_nwc_series()

    def estimate_nwc_coef(self):
        """
        한국 BS는 유동부채 컬럼 없음 → 보완:
          NWC ≈ (Receivables + Inventories) - (Short-term debt + Current LT debt + Lease)
        NWC/Sales 비율 OLS 또는 median.

        ★ v8: 마지막 실측 NWC 를 self._last_actual_nwc 에 저장 → ΔNWC 시드값
              (γ × last_actual_sales) 와의 정합성 진단용.
        """
        nwc_df = self._compute_bs_nwc_series()

        # ★ v8: 마지막 실측 NWC 저장
        self._last_actual_nwc = (float(nwc_df["nwc"].iloc[-1])
                                 if not nwc_df.empty else 0.0)

        sales = self._fs_wide["revenue"]
        sales_df = pd.DataFrame({"date": sales.index, "rev": sales.values}).reset_index(drop=True)
        merged = sales_df.merge(nwc_df, on="date", how="inner").dropna()
        merged = merged[merged["rev"] > 0]
        if merged.empty:
            self.report.add("nwc", "fallback_zero", n_obs=0, value=0.0,
                            note="merged 비어있음")
            return 0.0, "empty"

        slope, r2, n = self._ols_ratio(merged["rev"], merged["nwc"])
        use_ols = (not np.isnan(slope) and r2 >= OLS_MIN_R2
                   and n >= OLS_MIN_SAMPLES)
        if use_ols:
            self.report.add("nwc", "ok", n_obs=n, value=slope, r2=r2,
                            note="(Recv+Inv) - (ST debts) ratio")
            return float(slope), "ols"
        ratios = (merged["nwc"] / merged["rev"]).replace(
            [np.inf, -np.inf], np.nan).dropna()
        if ratios.empty:
            self.report.add("nwc", "fallback_zero", n_obs=0, value=0.0)
            return 0.0, "ratio_empty"
        med = float(self._winsorize(ratios).median())
        self.report.add("nwc", "fallback_median", n_obs=len(ratios),
                        value=med, r2=r2,
                        note=f"OLS R²={r2:.2f} → median ratio")
        return med, "median_ratio"

    # ─────────────────────────────────────────────────────────
    # 3. FCFF 계산
    # ─────────────────────────────────────────────────────────

    def compute_fcff(self):
        """Sales-driven FCFF = NOPAT + D&A - CapEx - ΔNWC.

        ★ v8 패치 (vs v1):
          v1: prev_nwc=None → 첫 예측분기 ΔNWC=0 → FCFF 과대계상
          v2: prev_nwc = γ × last_actual_sales → 첫 분기 ΔNWC 도 정합
        """
        sales_fc = self._sales_forecast
        tax      = self.estimate_tax_rate()
        opm_fc   = self.estimate_opm(sales_fc)

        alpha, _ = self.estimate_ratio_coef(
            ["da_cf", "intangible_amort_cf"], "da")
        beta_, _ = self.estimate_ratio_coef(
            ["capex_tangible", "capex_intangible"], "capex", take_abs=True)
        gamma, _ = self.estimate_nwc_coef()
        self._nwc_to_sales = gamma   # ★ v7: 결과 저장용

        # ★ v8: ΔNWC 시드값 = γ × 마지막 실제 매출 (v1: None)
        last_actual_sales = (float(self._sales_actual.iloc[-1])
                             if (self._sales_actual is not None
                                 and not self._sales_actual.empty)
                             else 0.0)
        prev_nwc = gamma * last_actual_sales

        # ★ v8: 실측 NWC 와 γ×sales 모델의 정합성 진단 로그
        if self.verbose and abs(last_actual_sales) > 0:
            actual_nwc = self._last_actual_nwc
            if actual_nwc is not None and abs(prev_nwc) > 0:
                ratio_gap = abs(actual_nwc - prev_nwc) / max(abs(actual_nwc), 1)
                if ratio_gap > 0.5:
                    log(self.ticker_dg,
                        f"[NWC seed] γ×sales={prev_nwc/1e9:,.1f}B vs "
                        f"실측={actual_nwc/1e9:,.1f}B 괴리 {ratio_gap:.1%}")

        rows = []
        for i, (dt, sales) in enumerate(sales_fc.items()):
            opm   = float(opm_fc.iloc[i])
            ebit  = sales * opm
            nopat = ebit * (1 - tax)
            da    = alpha * sales
            capex = beta_ * sales
            nwc   = gamma * sales
            delta_nwc = nwc - prev_nwc
            prev_nwc = nwc

            fcff = nopat + da - capex - delta_nwc
            invested = capex + max(delta_nwc, 0)
            roic = nopat / invested if invested > 1e-6 else np.nan
            reinv = capex / nopat if nopat > 1e-6 else np.nan

            rows.append({
                "date": dt,
                "quarter": f"{dt.year}Q{dt.quarter}",
                "sales_forecast": sales, "opm_forecast": opm,
                "ebit": ebit, "tax_rate": tax, "nopat": nopat,
                "da": da, "capex": capex, "nwc": nwc,
                "delta_nwc": delta_nwc, "fcff": fcff, "roic": roic,
                "reinvestment_rate": reinv,
            })
        self.result_df = pd.DataFrame(rows)
        self._compute_historical_fcff(tax, alpha, beta_, gamma)
        return self

    def _compute_historical_fcff(self, tax, alpha, beta_, gamma):
        """과거 FCFF 시계열 (terminal CAGR / 시각화용).

        ★ v8 패치: 첫 분기 ΔNWC = γ × first_sales 시드값
                   (v1: None → 첫 분기 ΔNWC=0 → CAGR 왜곡)
        """
        sales_act = self._sales_actual
        if sales_act is None or sales_act.empty:
            self._fcff_history = pd.Series(dtype=float); return

        wide = self._fs_wide
        rows = []
        # ★ v8: 첫 분기 ΔNWC 시드값 = γ × 첫 sales
        first_sales = float(sales_act.iloc[0]) if not sales_act.empty else 0.0
        prev_nwc = gamma * first_sales

        for dt, sales in sales_act.items():
            if dt not in wide.index:
                continue
            opm_row = wide.loc[dt].get("operating_income", np.nan)
            rev_row = wide.loc[dt].get("revenue", np.nan)
            if pd.isna(opm_row) or pd.isna(rev_row) or rev_row <= 0:
                prev_nwc = gamma * sales; continue
            opm   = float(opm_row) / float(rev_row)
            ebit  = sales * opm
            nopat = ebit * (1 - tax)
            da    = alpha * sales
            capex = beta_ * sales
            nwc   = gamma * sales
            dnwc  = nwc - prev_nwc
            prev_nwc = nwc
            rows.append({"date": dt, "fcff": nopat + da - capex - dnwc})
        if rows:
            self._fcff_history = pd.DataFrame(rows).set_index("date")["fcff"]
        else:
            self._fcff_history = pd.Series(dtype=float)

    # ─────────────────────────────────────────────────────────
    # 4. WACC 계산 (한국 시장 어댑테이션) — v1 동일
    # ─────────────────────────────────────────────────────────

    def compute_beta_re(self):
        """
        베타 + Re (CAPM):
          β_blume = 0.67×|β| + 0.33  (호영님 요청)
          Re      = Rf + β_blume × ERP
        """
        info = compute_beta_10y(
            self.ticker_dg, self.db_info, kospi_series=self.kospi,
            years=10, min_obs=750,
        )
        self._beta_info = info

        if np.isnan(info["beta_raw"]):
            beta_blume = 1.0  # fallback
            self.report.add("beta", "fallback_median", n_obs=info["n_obs"],
                            value=beta_blume,
                            note="베타 계산 실패 → β=1.0 fallback")
        else:
            beta_blume = info["beta_blume"]
            self.report.add("beta", "ok", n_obs=info["n_obs"],
                            value=beta_blume, r2=info["r_squared"],
                            note=f"β_raw={info['beta_raw']:.3f} → blume={beta_blume:.3f}")

        re = self.rf + beta_blume * self.erp
        if self.verbose:
            log(self.ticker_dg,
                f"β_raw={info.get('beta_raw', np.nan):.3f} "
                f"β_blume={beta_blume:.3f}  Re={re:.4%}  "
                f"(n={info['n_obs']}, R²={info.get('r_squared', np.nan):.2f})")
        return float(re), beta_blume

    def compute_cost_of_debt(self, tax):
        """이자비용 / 평균 총부채. 실패 시 RD_DEFAULT."""
        wide = self._fs_wide
        if "interest_expense" not in wide.columns:
            self.report.add("rd", "fallback_zero", value=RD_DEFAULT,
                            note="interest_expense 컬럼 없음")
            return RD_DEFAULT
        # 총부채 = 단기차입금 + 유동성장기부채 + 사채 + 장기차입금 + 리스
        total_debt = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                total_debt = total_debt + wide[k].fillna(0)
        td_avg = (total_debt + total_debt.shift(1)) / 2
        ie = wide["interest_expense"].abs()

        valid = (td_avg > 0) & ie.notna()
        if valid.sum() < 4:
            self.report.add("rd", "fallback_zero", n_obs=int(valid.sum()),
                            value=RD_DEFAULT, note="유효 분기 < 4")
            return RD_DEFAULT
        rd_series = (ie[valid] / td_avg[valid]).clip(0, 0.20)
        rd = float(rd_series.median())
        # ★ v8: Rd 하한 = Rf — 장부 이자율(변동금리·구채권 효과)이 무위험수익률보다
        #        낮으면 forward-looking 부채비용으로 부적절 (채권자도 최소 Rf 요구)
        if V8_RD_FLOOR_RF and rd < self.rf:
            self.report.add("rd", "ok", n_obs=int(valid.sum()), value=float(self.rf),
                            note=f"median {rd:.3%} < Rf → Rf floor {self.rf:.3%}")
            return float(self.rf)
        self.report.add("rd", "ok", n_obs=int(valid.sum()), value=rd)
        return rd

    def compute_wacc(self, re, tax):
        """WACC = Re × E/V + Rd × (1-t) × D/V"""
        rd = self.compute_cost_of_debt(tax)

        # E (시가총액)
        mc, mc_date = load_korea_marketcap_latest(
            self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0:
            # fallback: 현재가 × (자사주차감) 평균발행주식수
            wide = self._fs_wide
            cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
            shares_ser = wide.get("shares_treasury_adj", pd.Series([np.nan])).dropna()
            shares = shares_ser.iloc[-1] if not shares_ser.empty else np.nan
            if cp and not np.isnan(shares) and shares > 0:
                mkt_cap_won = cp * shares
                self.report.warn(f"시가총액 DB 없음 → 현재가×주식수={mkt_cap_won/1e12:.2f}조원 fallback")
            else:
                self.report.add("market_cap", "missing", n_obs=0,
                                note="시가총액·주가·주식수 모두 없음")
                if self.verbose: log(self.ticker_dg, "WACC: E 측정 실패 → Re만 사용")
                return float(re), re, rd  # WACC 대신 Re
        else:
            mkt_cap_won = mc * MARKETCAP_UNIT_MULTIPLIER  # 백만원 → 원

        # D (총부채)
        wide = self._fs_wide
        total_debt = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                total_debt = total_debt + wide[k].fillna(0)
        if total_debt.dropna().empty:
            d_won = 0.0
        else:
            d_won = float(total_debt.sort_index().iloc[-1])

        V = mkt_cap_won + d_won
        if V <= 0:
            self.report.add("wacc_components", "missing", value=0.0)
            return float(re), re, rd

        we = mkt_cap_won / V
        wd = d_won / V
        wacc = re * we + rd * (1 - tax) * wd
        # ★ v8: WACC 하한 = max(절대하한, Rf+1%) — 절대 floor 가 고레버리지 기업
        #        WACC 를 Rf 이하로 떨어뜨려 TV 발산을 유발하던 문제 차단
        wacc_lo = max(V8_WACC_FLOOR_ABS, self.rf + 0.01)
        wacc = float(np.clip(wacc, wacc_lo, WACC_CAP))

        self._mkt_cap = mkt_cap_won
        if self.verbose:
            log(self.ticker_dg,
                f"WACC={wacc:.4%}  Re={re:.4%} Rd={rd:.4%} tax={tax:.4%}  "
                f"E/V={we:.2%} D/V={wd:.2%}  E={mkt_cap_won/1e12:.2f}조 D={d_won/1e12:.2f}조")
        return wacc, re, rd

    # ─────────────────────────────────────────────────────────
    # 5. EVA / Moat / Phase 2 (★ v9.2 패치 적용)
    # ─────────────────────────────────────────────────────────

    def _compute_eva_spread(self, wacc):
        """ROIC_TTM - WACC. (US v7 _compute_eva_spread 한국 어댑테이션)"""
        wide = self._fs_wide
        if "operating_income" not in wide.columns or "total_equity" not in wide.columns:
            return {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan,
                    "n_positive": 0, "eva_series": []}

        tax = self.estimate_tax_rate()
        df = wide.copy()
        df["nopat_q"] = df["operating_income"] * (1 - tax)

        # IC = Equity + Total Debt - Cash
        td = pd.Series(0.0, index=df.index)
        for k in DEBT_KEYS:
            if k in df.columns:
                td = td + df[k].fillna(0)
        cash = pd.Series(0.0, index=df.index)
        for k in CASH_KEYS:
            if k in df.columns:
                cash = cash + df[k].fillna(0)
        df["ic"] = df["total_equity"].fillna(0) + td - cash

        df = df[["nopat_q", "ic"]].dropna().sort_index()
        if len(df) < 4:
            return {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan,
                    "n_positive": 0, "eva_series": []}

        eva_series = []
        dates = sorted(df.index)
        for i, dt in enumerate(dates):
            if i < 3: continue
            nopat_ttm = float(df["nopat_q"].iloc[i-3:i+1].sum())
            ic_snap = float(df["ic"].iloc[i])
            if ic_snap <= 0: continue
            roic_q = nopat_ttm / ic_snap
            eva_q = roic_q - wacc
            eva_series.append({"date": dt, "roic": roic_q,
                               "eva_spread": eva_q, "ic": ic_snap})

        if not eva_series:
            return {"roic": np.nan, "eva_spread": np.nan, "ic": np.nan,
                    "n_positive": 0, "eva_series": []}
        latest = eva_series[-1]
        recent_20 = eva_series[-20:]
        n_positive = sum(1 for e in recent_20 if e["eva_spread"] > 0)

        if self.verbose:
            log(self.ticker_dg,
                f"EVA: ROIC={latest['roic']:.2%}  WACC={wacc:.2%}  "
                f"spread={latest['eva_spread']:+.2%}  "
                f"n_pos(20Q)={n_positive}/20")

        return {
            "roic": latest["roic"], "eva_spread": latest["eva_spread"],
            "ic": latest["ic"], "n_positive": n_positive,
            "eva_series": eva_series,
        }

    def _moat_to_rho_and_years(self, eva):
        """
        EVA spread → (ρ, Phase2 기간, label).

        등급 기준 (US v11 동일, 한국에도 동일하게 적용):
          Wide moat   : spread > 15% AND 최근 20Q 중 15Q+ 양수 → ρ=0.90, 15년
          Narrow moat : spread  8~15% AND 12Q+ 양수            → ρ=0.83, 12년
          Some moat   : spread  3~8%  AND  8Q+ 양수            → ρ=0.75,  8년
          No moat     : spread < 3%   OR   < 8Q 양수           → ρ=0.60,  5년

        근거: Mauboussin & Johnson (1997) FAJ — Competitive Advantage Period
              Damodaran (2002) Ch.12 — extraordinary growth Phase2 권고
              Chan, Karceski, Lakonishok (2003) JF — 성장 지속성 실증
        """
        if np.isnan(eva.get("eva_spread", np.nan)):
            return 0.75, 8, "Unknown (fallback)"
        s, n = eva["eva_spread"], eva["n_positive"]
        if   s > 0.15 and n >= 15: rho, yrs, lbl = 0.90, 15, "Wide moat"
        elif s > 0.08 and n >= 12: rho, yrs, lbl = 0.83, 12, "Narrow moat"
        elif s > 0.03 and n >=  8: rho, yrs, lbl = 0.75,  8, "Some moat"
        else:                       rho, yrs, lbl = 0.60,  5, "No moat"

        if self.verbose:
            log(self.ticker_dg,
                f"Moat: [{lbl}]  spread={s:+.2%}  n_pos={n}/20  "
                f"→ ρ={rho}  Phase2={yrs}yr")
        return rho, yrs, lbl

    # ═══════════════════════════════════════════════════════════════
    # ★ v9.2 패치: _estimate_phase2_growth — 다중 fallback + min(hist, fc)
    # ═══════════════════════════════════════════════════════════════

    def _estimate_phase2_growth(self, wacc):
        """
        Phase 2 AR(1) 성장 경로 — 음수 g₀ 허용 + 다중 fallback (★ v9.2).

        v1 → v2 변경:
          v1: 8Q-CAGR (양수 케이스만) → 실패시 forecast yr1/yr2,
              하한 G_TERM(0.025) 으로 음수 g₀ 거부.
          v2: 1단계 g0_hist (8Q-CAGR → 12Q-TTM 순) +
              2단계 g0_fc (forecast yr1/yr2, 항상 계산) +
              3단계 결합 로직:
                · 양수×양수: min(hist, fc) → 보수적 채택
                · forecast 음수: forecast 우선 → 사이클 침체 신호 신뢰
                · 부호 불일치: 더 보수적인 (작은) 값
              하한 G_FLOOR_MIN(-0.20) 으로 침체기 표현 가능

        반환: (g_path, moat_label, rho, n_years)
        """
        G_TERM = self.gdp_growth   # 한국 0.025

        h = self._fcff_history

        # ════════════════════════════════════════════════════════
        # 1단계: 과거 시계열 기반 g₀ (g0_hist)
        # ════════════════════════════════════════════════════════
        g0_hist = None
        g0_hist_method = None

        # ── Method 1: 8Q CAGR (양수 케이스만) ─────────────────────
        if h is not None and len(h.dropna()) >= 8:
            h_c = h.dropna()
            f0, fl = float(h_c.iloc[-8]), float(h_c.iloc[-1])
            if f0 > 0 and fl > 0:
                cagr_q = (fl / f0) ** (4.0 / 8) - 1
                g0_hist = float(np.clip((1 + cagr_q) ** 4 - 1, G_FLOOR_MIN, G_CEIL))
                g0_hist_method = "8Q-CAGR"

        # ── Method 2: 12Q rolling TTM vs TTM-2 (★ v9.2 신규) ────
        if g0_hist is None and h is not None and len(h.dropna()) >= 12:
            h_c = h.dropna()
            ttm_recent = float(h_c.iloc[-4:].sum())
            ttm_prior  = float(h_c.iloc[-8:-4].sum())
            if abs(ttm_prior) > 1e-6:
                if ttm_prior > 0 and ttm_recent > 0:
                    g0_hist = ttm_recent / ttm_prior - 1
                elif ttm_prior > 0 and ttm_recent <= 0:
                    g0_hist = ttm_recent / ttm_prior - 1   # 자연스럽게 음수
                elif ttm_prior < 0 and ttm_recent < 0:
                    g0_hist = (ttm_recent - ttm_prior) / abs(ttm_prior)
                else:                                       # 적자→흑자
                    g0_hist = G_TERM                        # 보수적
                g0_hist = float(np.clip(g0_hist, G_FLOOR_MIN, G_CEIL))
                g0_hist_method = "12Q-TTM"

        # ════════════════════════════════════════════════════════
        # 2단계: Phase 1 forecast 기반 g₀ (g0_fc) — 항상 계산 ★
        # ════════════════════════════════════════════════════════
        g0_fc = None
        g0_fc_method = None

        if self.result_df is not None and not self.result_df.empty:
            fcff_q = self.result_df["fcff"].values
            if len(fcff_q) >= 4:
                yr1 = float(np.sum(fcff_q[:4]))
                yr2 = float(np.sum(fcff_q[4:8])) if len(fcff_q) >= 8 else yr1
                if abs(yr1) > 1e-6:
                    if yr1 > 0 and yr2 > 0:
                        g0_fc = yr2 / yr1 - 1
                    elif yr1 > 0 and yr2 <= 0:
                        g0_fc = yr2 / yr1 - 1               # 자연스럽게 음수
                    elif yr1 < 0 and yr2 < 0:
                        g0_fc = (yr2 - yr1) / abs(yr1)
                    else:                                   # 적자→흑자
                        g0_fc = G_TERM
                    g0_fc = float(np.clip(g0_fc, G_FLOOR_MIN, G_CEIL))
                    g0_fc_method = "FC-yr1-yr2"

        # ════════════════════════════════════════════════════════
        # 3단계: g0_hist 와 g0_fc 결합 (★ v9.2 핵심 로직)
        # ════════════════════════════════════════════════════════
        if g0_hist is not None and g0_fc is not None:
            # 두 값 모두 양수 → min 채택 (보수적)
            if g0_hist > 0 and g0_fc > 0:
                if g0_hist > g0_fc:
                    g0 = g0_fc
                    g0_method = f"min({g0_hist_method}={g0_hist*100:.1f}%, FC={g0_fc*100:.1f}%) → FC"
                else:
                    g0 = g0_hist
                    g0_method = f"min({g0_hist_method}={g0_hist*100:.1f}%, FC={g0_fc*100:.1f}%) → {g0_hist_method}"
            # forecast 가 음수 → forecast 우선
            elif g0_fc <= 0:
                g0 = g0_fc
                g0_method = f"FC-priority (FC={g0_fc*100:.1f}% ≤ 0, hist={g0_hist*100:.1f}%)"
            # forecast 양수, hist 음수 → 더 보수적인 (작은) 값
            else:
                g0 = min(g0_hist, g0_fc)
                g0_method = f"min(hist={g0_hist*100:.1f}%, FC={g0_fc*100:.1f}%) → 부호 불일치 보수"
        elif g0_hist is not None:
            g0 = g0_hist
            g0_method = g0_hist_method
        elif g0_fc is not None:
            g0 = g0_fc
            g0_method = g0_fc_method
        else:
            # 최후 fallback
            g0 = G_TERM
            g0_method = "default-G_TERM"

        # ── EVA → 해자 등급 → (ρ, 기간) ────────────────────────────
        eva = self._compute_eva_spread(wacc)
        rho, n_years, moat_label = self._moat_to_rho_and_years(eva)

        # ── ρ OLS 보정 (★ v9.2: 양수 조건 제거 — 음수 annual 도 포함) ──
        if h is not None and len(h.dropna()) >= 12:
            h_c = h.dropna()
            annual_vals = [float(h_c.iloc[y:y+4].sum())
                           for y in range(0, len(h_c) - len(h_c) % 4, 4)]
            if len(annual_vals) >= 4:
                g_hist = [(annual_vals[i] / annual_vals[i-1] - 1)
                          if abs(annual_vals[i-1]) > 1e-6 else 0.0
                          for i in range(1, len(annual_vals))]
                if len(g_hist) >= 3:
                    yt = np.array([g - G_TERM for g in g_hist[1:]])
                    xt = np.array([g - G_TERM for g in g_hist[:-1]])
                    if np.dot(xt, xt) > 1e-10:
                        rho_ols = float(np.clip(
                            np.dot(xt, yt) / np.dot(xt, xt),
                            0.40, 0.92,
                        ))
                        # EVA 기반 ρ 와 OLS ρ 의 가중 평균 (각 50%)
                        rho = float(np.clip(0.5 * rho + 0.5 * rho_ols, 0.40, 0.92))
                        if self.verbose:
                            log(self.ticker_dg,
                                f"ρ 보정: EVA기반={rho:.3f}  OLS={rho_ols:.3f}  "
                                f"→ 혼합={rho:.3f}")

        # ── ★ v8: g₀ 기반 ρ 적응형 상한 (고성장일수록 빠른 평균회귀) ──
        #     g0>30% → ρ≤0.75(hyper) / >15% → ρ≤0.85(high) / ≤15% → ρ≤0.92(normal)
        #     근거: 고성장은 지속성이 낮음 → 평균회귀 가속 (Chan/Karceski/Lakonishok 2003)
        rho_cap_applied = False
        if V8_RHO_ADAPTIVE_CAP:
            if   g0 > 0.30: rho_cap = 0.75
            elif g0 > 0.15: rho_cap = 0.85
            else:           rho_cap = 0.92
            if rho > rho_cap:
                if self.verbose:
                    log(self.ticker_dg,
                        f"[v8] ρ 적응형 상한: g0={g0:.1%}  ρ {rho:.3f}→{rho_cap:.3f}")
                rho = rho_cap
                rho_cap_applied = True

        # ── AR(1) 성장 경로 — clip 하한 G_FLOOR_MIN 으로 완화 ────
        g = g0
        g_path = []
        for _ in range(n_years):
            g = G_TERM + (g - G_TERM) * rho
            g_path.append(float(np.clip(g, G_FLOOR_MIN, G_CEIL)))

        if self.verbose:
            log(self.ticker_dg,
                f"Phase2 [{moat_label}] ρ={rho:.3f} {n_years}년  "
                f"g0={g0:.1%} ({g0_method}) → {g_path[0]:.1%} ... {g_path[-1]:.1%}")

        # EVA 정보 + g₀ 진단 정보 캐싱 (export / decomposition 셀에서 사용)
        self._eva_cache = {
            "eva_spread":   eva.get("eva_spread", np.nan),
            "roic":         eva.get("roic",        np.nan),
            "n_positive":   eva.get("n_positive",  0),
            "moat_label":   moat_label,
            "rho":          rho,
            "n_phase2":     n_years,
            "eva_series":   eva.get("eva_series",  []),
            # ★ v9.2: g₀ 진단 정보
            "g0":           g0,
            "g0_method":    g0_method,
            "g0_hist":      g0_hist,
            "g0_hist_method": g0_hist_method,
            "g0_fc":        g0_fc,
            "g0_fc_method": g0_fc_method,
            "rho_cap_applied": rho_cap_applied,   # ★ v8
        }
        return g_path, moat_label, rho, n_years

    # ─────────────────────────────────────────────────────────
    # ★ v1.5: Phase 2 미드사이클 OPM 수렴
    # ─────────────────────────────────────────────────────────
    def _apply_midcycle(self, ph2_annual: np.ndarray, ph2_growth: List[float],
                        rho: float, tax: float) -> Dict:
        """Phase 1 말 OPM(최근 4Q 가중) → 과거 미드사이클 OPM 으로 Phase 2 동안 선형 수렴.
          FCFF_adj(t) = FCFF(t) + Sales(t) × (OPM(t) − OPM_end) × (1 − T)
          Sales(t): Phase1 연간매출에서 매출 g₀(yr2/yr1)를 FCFF 와 같은 ρ 로 g_term 까지 감쇠
        """
        out = {"enabled": bool(MIDCYCLE_ENABLE), "applied": False, "ph2_annual": ph2_annual,
               "reason": None}
        if not MIDCYCLE_ENABLE or len(ph2_annual) == 0:
            out["reason"] = "disabled"
            return out
        df = self.result_df
        n4 = min(4, len(df))
        s_end = float(df["sales_forecast"].iloc[-n4:].sum())
        opm_end = float(df["ebit"].iloc[-n4:].sum() / s_end) if s_end > 0 else np.nan

        hist = self._opm_hist_frame().tail(MIDCYCLE_WINDOW_Q)
        if len(hist) < 12 or not np.isfinite(opm_end):
            out["reason"] = f"미드사이클 표본 부족 ({len(hist)}Q)"
            return out
        if MIDCYCLE_STAT == "weighted":
            opm_mid = float(hist["operating_income"].sum() / hist["revenue"].sum())
        else:
            opm_mid = float(hist["opm"].median())
        gap = opm_mid - opm_end
        out.update({"opm_end": opm_end, "opm_mid": opm_mid, "mid_stat": MIDCYCLE_STAT,
                    "mid_window": f"{hist['per'].iloc[0]}~{hist['per'].iloc[-1]} ({len(hist)}Q)"})
        if abs(gap) < MIDCYCLE_MIN_GAP:
            out["reason"] = f"|OPM_end − OPM_mid| < {MIDCYCLE_MIN_GAP:.0%}"
            return out
        if MIDCYCLE_DIRECTION == "down_only" and gap > 0:
            out["reason"] = "down_only: Phase1 말 OPM 이 미드사이클보다 낮음 → 미적용"
            return out

        # 매출 경로 (Phase 2)
        sq = df["sales_forecast"].values
        yr1 = float(sq[:4].sum()); yr2 = float(sq[4:8].sum()) if len(sq) >= 8 else yr1
        G = self.gdp_growth
        gs = float(np.clip(yr2 / yr1 - 1, G_FLOOR_MIN, G_CEIL)) if yr1 > 0 else G
        n = len(ph2_annual)
        ny = int(MIDCYCLE_YEARS) if MIDCYCLE_YEARS else n
        ny = max(1, min(ny, n))
        sales_path, opm_path, adj = [], [], []
        s = s_end
        for k in range(1, n + 1):
            gs = float(np.clip(G + (gs - G) * rho, G_FLOOR_MIN, G_CEIL))
            s = s * (1 + gs)
            w = min(k / ny, 1.0)
            o = opm_end + gap * w
            sales_path.append(s); opm_path.append(o)
            adj.append(s * (o - opm_end) * (1 - tax))
        ph2_new = ph2_annual + np.array(adj)
        out.update({"applied": True, "years": ny, "sales_path": sales_path, "opm_path": opm_path,
                    "adjustment": adj, "ph2_annual_pre": ph2_annual.tolist(),
                    "ph2_annual": ph2_new, "neg_terminal": bool(ph2_new[-1] <= 0 < ph2_annual[-1])})
        if self.verbose:
            log(self.ticker_dg, f"[Mid-cycle] OPM {opm_end:.1%} → {opm_mid:.1%} "
                f"({MIDCYCLE_STAT}, {out['mid_window']}) {ny}년 선형 수렴 | "
                f"Phase2 말 FCFF {ph2_annual[-1]/1e12:,.1f}조 → {ph2_new[-1]/1e12:,.1f}조")
        if out["neg_terminal"]:
            log(self.ticker_dg, "[경고] [Mid-cycle] 조정 후 Phase2 말 FCFF ≤ 0 → TV 음수. "
                "MIDCYCLE_STAT/WINDOW 또는 매출 경로 점검 권장")
        return out

    def compute_terminal_growth(self, reinv, wacc):
        """g_terminal ∈ [0, GDP_GROWTH] — Phase 2 에서 이미 수렴 처리되므로 보수적."""
        fcff_cagr = 0.0
        h = self._fcff_history
        if h is not None and len(h) >= 4:
            h20 = h.dropna().tail(20)
            f0, fl = float(h20.iloc[0]), float(h20.iloc[-1])
            n = len(h20)
            if f0 > 0 and fl > 0 and n >= 4:
                fcff_cagr = (fl / f0) ** (4.0 / n) - 1.0
                fcff_cagr = float(np.clip(fcff_cagr, -0.10, 0.15))
        roic_med = (float(self.result_df["roic"].dropna().median())
                    if not self.result_df["roic"].dropna().empty else 0.06)
        g_roic = roic_med * reinv
        # ★ v8: 영구성장률 상한 = min(GDP, Rf) (Damodaran: g_terminal ≤ 무위험수익률)
        g_cap = min(self.gdp_growth, self.rf) if V8_TERMINAL_RF_CAP else self.gdp_growth
        g = float(np.clip(0.5 * fcff_cagr + 0.5 * g_roic, 0.0, g_cap))

        if self.verbose:
            log(self.ticker_dg,
                f"g_terminal={g:.4f}  FCFF_CAGR={fcff_cagr:.4f}  g_roic={g_roic:.4f}")
        return g

    # ─────────────────────────────────────────────────────────
    # 6. Valuation — 3-Stage DCF
    # ─────────────────────────────────────────────────────────

    def compute_valuation(self):
        re, beta_blume = self.compute_beta_re()
        tax = self.estimate_tax_rate()
        wacc, re_used, rd = self.compute_wacc(re, tax)

        df = self.result_df.copy()
        # reinvestment_rate median
        pos = df[df["nopat"] > 0]["reinvestment_rate"].dropna()
        reinv_med = float(self._winsorize(pos).median()) if not pos.empty else 0.3

        # Phase 1: 분기 → 연간
        fcff_q = df["fcff"].values
        ph1_annual = np.array([float(np.sum(fcff_q[yr:yr+4]))
                               for yr in range(0, len(fcff_q), 4)])
        T_ph1 = len(ph1_annual)

        # Phase 2: AR(1)  ★ v9.2
        ph2_growth, moat_lbl, rho, n_ph2 = self._estimate_phase2_growth(wacc)
        ph2_annual = []
        last = float(ph1_annual[-1])
        for g in ph2_growth:
            last = last * (1 + g)
            ph2_annual.append(last)
        ph2_annual = np.array(ph2_annual)

        # ★ v1.5: Phase 2 미드사이클 OPM 수렴 ─────────────────────────
        ph2_annual_pre = ph2_annual.copy()
        midcycle = self._apply_midcycle(ph2_annual, ph2_growth, rho, tax)
        ph2_annual = midcycle["ph2_annual"]

        # Phase 3: TV  (★ v8 보수화 — 최소 스프레드 2% + TV 배수 상한)
        g_term = self.compute_terminal_growth(reinv_med, wacc)
        # ★ v8: 최소 스프레드 2% (구 1% → TV배수 100배+ 발산 차단)
        if wacc - g_term < V8_MIN_TV_SPREAD:
            g_term = max(0.0, wacc - V8_MIN_TV_SPREAD)
        fcff_last_annual = float(ph2_annual[-1])
        tv = fcff_last_annual * (1 + g_term) / (wacc - g_term)
        # ★ v8: TV / 마지막 연간 FCFF 배수 상한 (양(+) FCFF 에만, 이중 안전장치)
        tv_capped = False
        tv_fcff_mult = np.nan
        if fcff_last_annual > 0:
            tv_fcff_mult = tv / fcff_last_annual
            if V8_TV_FCFF_MULT_CAP and tv > V8_TV_FCFF_MULT_CAP * fcff_last_annual:
                tv = V8_TV_FCFF_MULT_CAP * fcff_last_annual
                tv_fcff_mult = float(V8_TV_FCFF_MULT_CAP)
                tv_capped = True
                if self.verbose:
                    log(self.ticker_dg,
                        f"[v8] TV 배수 상한 {V8_TV_FCFF_MULT_CAP:.0f}× 적용 "
                        f"(spread={wacc - g_term:.2%})")

        # PV
        all_annual = np.concatenate([ph1_annual, ph2_annual])
        T = len(all_annual)
        pv_fcff = float(sum(all_annual[t] / (1 + wacc) ** (t+1) for t in range(T)))
        pv_tv = tv / (1 + wacc) ** T
        ev = pv_fcff + pv_tv

        # Net Debt (latest)
        wide = self._fs_wide
        td = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                td = td + wide[k].fillna(0)
        cash = pd.Series(0.0, index=wide.index)
        for k in CASH_KEYS:
            if k in wide.columns:
                cash = cash + wide[k].fillna(0)
        net_debt_series = (td - cash).dropna()
        net_debt = float(net_debt_series.iloc[-1]) if not net_debt_series.empty else 0.0

        equity_val = ev - net_debt

        # ── ★ v8: Sanity Guard — 산출 지분가치 vs 실제 시총 괴리 검사 ──
        #     입력오류(주식수/단위/FCFF)로 지분가치가 시총의 수천~수만 배가 되는
        #     사례를 차단. ValueError → process_one_ticker_kr 에서 fail 기록.
        mkt_cap_actual = self._mkt_cap if self._mkt_cap else np.nan
        mc_ratio = (equity_val / mkt_cap_actual
                    if (mkt_cap_actual and np.isfinite(mkt_cap_actual)
                        and mkt_cap_actual > 0)
                    else np.nan)
        if V8_SANITY_GUARD and np.isfinite(mc_ratio):
            if mc_ratio > V8_SANITY_MC_RATIO or (0 < mc_ratio < 1.0 / V8_SANITY_MC_RATIO):
                raise ValueError(
                    f"[{self.ticker_dg}] v8 sanity guard: 산출 지분가치 "
                    f"{equity_val/1e12:,.2f}조 vs 실제 시총 {mkt_cap_actual/1e12:,.2f}조 "
                    f"= {mc_ratio:,.1f}배 (허용 ±{V8_SANITY_MC_RATIO:.0f}배) → 평가 제외. "
                    f"입력(주식수/FCFF/NWC) 점검 필요")

        # Shares (자사주차감 우선)
        shares = np.nan
        for col in ["shares_treasury_adj", "shares_common"]:
            if col in wide.columns:
                s_ser = wide[col].dropna()
                s_ser = s_ser[s_ser > 0]
                if not s_ser.empty:
                    shares = float(s_ser.iloc[-1]); break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0,
                            note="주식수 컬럼 모두 결측")
            target_price = np.nan
        else:
            target_price = equity_val / shares

        # 현재가
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        self._current_price = cp
        upside = (((target_price/cp) - 1) * 100
                  if (cp and not np.isnan(target_price) and cp > 0) else np.nan)

        tv_weight = pv_tv / ev * 100 if ev != 0 else np.nan

        self.valuation = {
            "ticker": self.ticker_dg,
            "wacc": wacc, "wacc_re": re_used, "wacc_rd": rd,
            "beta_raw": self._beta_info.get("beta_raw") if self._beta_info else np.nan,
            "beta_blume": beta_blume,
            "g_terminal": g_term, "reinvestment_rate": reinv_med,
            "pv_fcff": pv_fcff, "terminal_value": tv, "pv_tv": pv_tv,
            "tv_weight_pct": tv_weight,
            "enterprise_value": ev, "net_debt": net_debt,
            "equity_value": equity_val, "shares": shares,
            "target_price": target_price, "current_price": cp,
            "upside_pct": upside,
            # 단계별 진단용 (Cell 7 / Cell 8 에서 사용)
            "ph1_annual": ph1_annual.tolist(),
            "ph2_annual": ph2_annual.tolist(),
            "ph2_growth": ph2_growth,
            "all_annual": all_annual.tolist(),
            "t_total":    T,
            "fcff_last_annual": fcff_last_annual,
            "annual_fcff": all_annual.tolist(),  # 구버전 호환 키
            # EVA / Moat / g₀ 정보
            "moat_label":   moat_lbl,
            "eva_spread":   self._eva_cache.get("eva_spread", np.nan),
            "roic":         self._eva_cache.get("roic", np.nan),
            "moat_rho":     rho,
            "rho_cap_applied": self._eva_cache.get("rho_cap_applied", False),  # ★ v8
            "tv_fcff_mult": tv_fcff_mult,   # ★ v8
            "tv_capped":    tv_capped,      # ★ v8
            "mc_ratio":     mc_ratio,       # ★ v8
            "n_phase2":     n_ph2,
            # ★ v1.5 OPM / 미드사이클 진단
            "opm_info":         dict(getattr(self, "_opm_info", {})),
            "midcycle":         {k: (v.tolist() if isinstance(v, np.ndarray) else v)
                                 for k, v in midcycle.items()},
            "ph2_annual_pre_midcycle": ph2_annual_pre.tolist(),
            "eva_series":   self._eva_cache.get("eva_series", []),
            "g0":           self._eva_cache.get("g0", np.nan),
            "g0_method":    self._eva_cache.get("g0_method", "?"),
            "g0_hist":      self._eva_cache.get("g0_hist"),
            "g0_fc":        self._eva_cache.get("g0_fc"),
            # 한국 메타데이터
            "nwc_method":        getattr(self, "_nwc_method", "?"),
            "nwc_to_sales":      getattr(self, "_nwc_to_sales", np.nan),
            "op_current_assets": getattr(self, "_op_ca_latest", np.nan),
            "op_current_liab":   getattr(self, "_op_cl_latest", np.nan),
            "revenue_quarters": len(self._sales_actual),
            "forecast_model":   self._used_model,
            "forecast_date":    self._forecast_date,
            "t_years":          T,
        }

        if self.verbose:
            tp_str = f"{target_price:,.0f}원" if not np.isnan(target_price) else "N/A"
            cp_str = f"{cp:,.0f}원" if cp else "N/A"
            up_str = f"{upside:+.1f}%" if not np.isnan(upside) else "N/A"
            log(self.ticker_dg,
                f"3-Stage  EV={ev/1e12:.2f}조  TP={tp_str}  CP={cp_str}  Up={up_str}  "
                f"TV비중={tv_weight:.1f}%  Moat={moat_lbl}")
        return self

    # ─────────────────────────────────────────────────────────
    # 7. DB 저장 (v1 동일)
    # ─────────────────────────────────────────────────────────

    def save_to_db(self, run_date=None):
        if self.result_df is None or self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation

        rows = []
        for _, row in self.result_df.iterrows():
            rows.append({
                "date": run_date, "ticker": self.ticker_dg,
                "quarter": row.get("quarter"),
                "sales_forecast": row.get("sales_forecast"),
                "opm_forecast": row.get("opm_forecast"),
                "ebit": row.get("ebit"), "tax_rate": row.get("tax_rate"),
                "nopat": row.get("nopat"), "da": row.get("da"),
                "capex": row.get("capex"), "nwc": row.get("nwc"),
                "delta_nwc": row.get("delta_nwc"), "fcff": row.get("fcff"),
                "roic": row.get("roic"),
                "reinvestment_rate": v["reinvestment_rate"],
                "g_terminal": v["g_terminal"],
                "discount_rate": v["wacc"],
                "wacc_re": v["wacc_re"], "wacc_rd": v["wacc_rd"],
                "beta_raw": v.get("beta_raw"), "beta_blume": v["beta_blume"],
                "enterprise_value": v["enterprise_value"],
                "net_debt": v["net_debt"], "equity_value": v["equity_value"],
                "shares": v["shares"], "target_price": v["target_price"],
                "current_price": v["current_price"], "upside_pct": v["upside_pct"],
                "moat_label": v["moat_label"], "eva_spread": v["eva_spread"],
                "moat_rho": v.get("moat_rho"),
                "rho_cap_applied": int(bool(v.get("rho_cap_applied", 0))),
                "tv_weight_pct": v.get("tv_weight_pct"),
                "tv_fcff_mult": v.get("tv_fcff_mult"),
                "tv_capped": int(bool(v.get("tv_capped", 0))),
                "mc_ratio": v.get("mc_ratio"),
                "nwc_method": v.get("nwc_method"),
                "nwc_to_sales": v.get("nwc_to_sales"),
                "op_current_assets": v.get("op_current_assets"),
                "op_current_liab": v.get("op_current_liab"),
                "revenue_quarters": v["revenue_quarters"],
                "forecast_model": v["forecast_model"],
                "forecast_date": v["forecast_date"],
            })

        save_df = pd.DataFrame(rows).where(pd.notnull(pd.DataFrame(rows)), None)
        # NaN/Inf → None
        for col in save_df.select_dtypes(include=[float]).columns:
            save_df[col] = save_df[col].apply(
                lambda x: None if (x is None or pd.isna(x) or np.isinf(x)) else x)

        sql = f'''
            INSERT INTO `{TABLE_RESULT}`
            (date, ticker, quarter, sales_forecast, opm_forecast, ebit, tax_rate,
             nopat, da, capex, nwc, delta_nwc, fcff, roic, reinvestment_rate,
             g_terminal, discount_rate, wacc_re, wacc_rd,
             beta_raw, beta_blume,
             enterprise_value, net_debt, equity_value,
             shares, target_price, current_price, upside_pct,
             moat_label, eva_spread,
             moat_rho, rho_cap_applied, tv_weight_pct, tv_fcff_mult, tv_capped, mc_ratio,
             nwc_method, nwc_to_sales, op_current_assets, op_current_liab,
             revenue_quarters,
             forecast_model, forecast_date)
            VALUES
            (%(date)s,%(ticker)s,%(quarter)s,%(sales_forecast)s,%(opm_forecast)s,
             %(ebit)s,%(tax_rate)s,%(nopat)s,%(da)s,%(capex)s,%(nwc)s,
             %(delta_nwc)s,%(fcff)s,%(roic)s,%(reinvestment_rate)s,
             %(g_terminal)s,%(discount_rate)s,%(wacc_re)s,%(wacc_rd)s,
             %(beta_raw)s,%(beta_blume)s,
             %(enterprise_value)s,%(net_debt)s,%(equity_value)s,
             %(shares)s,%(target_price)s,%(current_price)s,%(upside_pct)s,
             %(moat_label)s,%(eva_spread)s,
             %(moat_rho)s,%(rho_cap_applied)s,%(tv_weight_pct)s,%(tv_fcff_mult)s,%(tv_capped)s,%(mc_ratio)s,
             %(nwc_method)s,%(nwc_to_sales)s,%(op_current_assets)s,%(op_current_liab)s,
             %(revenue_quarters)s,
             %(forecast_model)s,%(forecast_date)s)
            ON DUPLICATE KEY UPDATE
                fcff=VALUES(fcff), target_price=VALUES(target_price),
                upside_pct=VALUES(upside_pct),
                enterprise_value=VALUES(enterprise_value),
                discount_rate=VALUES(discount_rate),
                g_terminal=VALUES(g_terminal),
                moat_rho=VALUES(moat_rho), rho_cap_applied=VALUES(rho_cap_applied),
                tv_weight_pct=VALUES(tv_weight_pct), tv_fcff_mult=VALUES(tv_fcff_mult),
                tv_capped=VALUES(tv_capped), mc_ratio=VALUES(mc_ratio)
        '''
        conn = get_pymysql_conn(db_info)
        try:
            with conn.cursor() as cur:
                cur.executemany(sql, save_df.to_dict("records"))
            conn.commit()
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()
        return len(rows)

    # ─────────────────────────────────────────────────────────
    # 8. 시각화 (v1 4-패널 미러 — 한국 단위 적용)
    # ─────────────────────────────────────────────────────────

    def plot(self):
        if self.result_df is None:
            print("[WARN] result_df 없음"); return

        df = self.result_df.copy()
        v = self.valuation or {}
        h = self._fcff_history

        # 단위: 조원 (1e12)
        UNIT, UNIT_LBL = 1e12, "조원"

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f"{self.ticker_dg}  Korea FCFF DCF Valuation v2",
                     fontsize=14, fontweight="bold")

        # ① 과거+예측 FCFF
        ax = axes[0, 0]
        if h is not None and not h.empty:
            h_plot = h.dropna().tail(16)
            h_lbl = [f"{d.year}Q{d.quarter}" for d in h_plot.index]
            ax.bar(range(len(h_plot)), h_plot.values/UNIT,
                   color=["#2980b9" if v_>=0 else "#c0392b" for v_ in h_plot.values],
                   alpha=0.75, label="Historical FCFF", edgecolor="white")
            ax.axvline(len(h_plot)-0.5, color="gray", lw=1.2, ls="--",
                       label="Forecast start")
            xt = list(range(len(h_plot))); xl = h_lbl; off = len(h_plot)
        else:
            xt, xl, off = [], [], 0
        fc_col = ["#27ae60" if f>=0 else "#e74c3c" for f in df["fcff"]]
        ax.bar(range(off, off+len(df)), df["fcff"].values/UNIT,
               color=fc_col, alpha=0.90, label="Forecast FCFF", edgecolor="white")
        xt += list(range(off, off+len(df))); xl += df["quarter"].tolist()
        ax.axhline(0, color="black", lw=0.8)
        ax.set_xticks(xt); ax.set_xticklabels(xl, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"Historical & Forecast FCFF ({UNIT_LBL})")
        ax.set_ylabel(UNIT_LBL); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # ② Sales + EBIT
        ax = axes[0, 1]
        act = self._sales_actual
        if act is not None and not act.empty:
            ap = act.tail(16); al = [f"{d.year}Q{d.quarter}" for d in ap.index]
            ax.plot(range(len(ap)), ap.values/UNIT, marker="o", ms=3, lw=1.5,
                    color="#2980b9", label="Actual Sales", zorder=3)
            ax.axvline(len(ap)-0.5, color="gray", lw=1.2, ls="--")
            so, sx, sl = len(ap), list(range(len(ap))), al
        else:
            so, sx, sl = 0, [], []
        ax.bar(range(so, so+len(df)), df["sales_forecast"].values/UNIT,
               alpha=0.45, label="Forecast Sales", color="#3498db")
        ax.bar(range(so, so+len(df)), df["ebit"].values/UNIT,
               alpha=0.70, label="Forecast EBIT", color="#e67e22")
        sx += list(range(so, so+len(df))); sl += df["quarter"].tolist()
        ax.set_xticks(sx); ax.set_xticklabels(sl, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"Actual Sales → Forecast Sales & EBIT ({UNIT_LBL})")
        ax.set_ylabel(UNIT_LBL); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # ③ OPM
        ax = axes[1, 0]
        wide = self._fs_wide
        if "operating_income" in wide.columns and "revenue" in wide.columns:
            opm_h = (wide["operating_income"]/wide["revenue"]).dropna().tail(16)
            ax.plot(range(len(opm_h)), opm_h.values*100, marker="o", ms=3, lw=1.5,
                    color="purple", label="Historical OPM")
            ax.axvline(len(opm_h)-0.5, color="gray", lw=1.2, ls="--")
            oo, ox, oxl = (len(opm_h), list(range(len(opm_h))),
                           [f"{d.year}Q{d.quarter}" for d in opm_h.index])
        else:
            oo, ox, oxl = 0, [], []
        ax.plot(range(oo, oo+len(df)), df["opm_forecast"].values*100,
                marker="s", ms=3, lw=1.5, ls="--", color="#8e44ad",
                label="Forecast OPM")
        ox += list(range(oo, oo+len(df))); oxl += df["quarter"].tolist()
        ax.set_xticks(ox); ax.set_xticklabels(oxl, rotation=45, ha="right", fontsize=7)
        ax.axhline(0, color="black", lw=0.8); ax.set_title("OPM (%)")
        ax.set_ylabel("%"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:.1f}%"))

        # ④ 요약
        ax = axes[1, 1]; ax.axis("off")
        def _fmt(x, unit="조", d=2):
            if pd.isna(x) or (isinstance(x,float) and np.isnan(x)): return "N/A"
            if unit == "조": return f"{x/1e12:,.{d}f}조원"
            if unit == "%":  return f"{x*100:.{d}f}%"
            return f"{x:,.{d}f}원"
        lines = [
            f"Current Price : {_fmt(v.get('current_price'),'원',0)}",
            f"Target Price  : {_fmt(v.get('target_price'),'원',0)}",
            f"Upside        : {v.get('upside_pct',0):+.1f}%" if not np.isnan(v.get('upside_pct',np.nan)) else "Upside : N/A",
            "─" * 32,
            f"PV(FCFF)      : {_fmt(v.get('pv_fcff'))}",
            f"Terminal Val  : {_fmt(v.get('terminal_value'))}",
            f"Enterprise V  : {_fmt(v.get('enterprise_value'))}",
            f"Net Debt      : {_fmt(v.get('net_debt'))}",
            f"Equity Value  : {_fmt(v.get('equity_value'))}",
            "─" * 32,
            f"WACC          : {_fmt(v.get('wacc'),'%')}",
            f"  Re          : {_fmt(v.get('wacc_re'),'%')}",
            f"  Rd          : {_fmt(v.get('wacc_rd'),'%')}",
            f"  β_blume     : {v.get('beta_blume',np.nan):.3f}",
            f"g_terminal    : {_fmt(v.get('g_terminal'),'%')}",
            "─" * 32,
            f"Moat          : {v.get('moat_label','N/A')}",
            f"EVA spread    : {_fmt(v.get('eva_spread'),'%')}",
            f"g₀ method     : {v.get('g0_method','?')}",
            f"NWC 정의      : {v.get('nwc_method','?')}",
            f"Quarters used : {v.get('revenue_quarters',0)}",
            f"Forecast model: {v.get('forecast_model','N/A')}",
        ]
        ax.text(0.05, 0.97, "\n".join(lines), transform=ax.transAxes,
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.8))
        plt.tight_layout(); plt.show()

    # ─────────────────────────────────────────────────────────
    # 9. 전체 실행
    # ─────────────────────────────────────────────────────────

    def run(self):
        self.load_sales()
        self.load_financials()
        self.compute_fcff()
        self.compute_valuation()
        return self


# ── 단일 티커 처리 래퍼 ──────────────────────────────────────────
def process_one_ticker_kr(
    ticker, engine, db_info, rf, e_rm, kospi_series,
    verbose=False, run_date=None, save_db=True,
):
    """KoreaDCFModel 실행 + 결과 + 데이터 품질 리포트 반환."""
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        m = KoreaDCFModel(ticker=ticker, engine=engine, db_info=db_info,
                          rf=rf, e_rm=e_rm, kospi_series=kospi_series,
                          verbose=verbose)
        m.run()
        rows = m.save_to_db(run_date) if save_db else 0
        v = m.valuation
        return {
            "status": "ok", "ticker": m.ticker_dg,
            "target_price":  v.get("target_price", np.nan),
            "current_price": v.get("current_price", np.nan),
            "upside_pct":    v.get("upside_pct", np.nan),
            "wacc":          v.get("wacc", np.nan),
            "g_terminal":    v.get("g_terminal", np.nan),
            "moat":          v.get("moat_label", "?"),
            "rows_saved":    rows,
            "report":        m.report,
            "msg": (f"TP={v.get('target_price',np.nan):,.0f}원"
                    if not np.isnan(v.get('target_price', np.nan)) else "TP=N/A"),
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": to_dg_ticker(ticker),
            "target_price": np.nan, "current_price": np.nan,
            "upside_pct": np.nan, "wacc": np.nan, "g_terminal": np.nan,
            "moat": "?", "rows_saved": 0,
            "report": None,
            "msg": str(e)[:120],
        }
    finally:
        clear_memory()


print("[OK] KoreaDCFModel v8 클래스 & process_one_ticker_kr 정의 완료")


[OK] KoreaDCFModel v8 클래스 & process_one_ticker_kr 정의 완료


## Cell 8 · Excel Export 모듈 import & 등록

`DATA/FCFF_Result_export_excel_module.py` (9-sheet Excel builder) 를 강제 리로드 후 `KoreaDCFModel` 에 `export_korea_excel` 메서드를 attach.

> ⚠️ Cell 7(클래스 셀)을 다시 실행하면 attach 가 풀리므로, 이 셀도 반드시 재실행하세요.

In [20]:
# ═══════════════════════════════════════════════════════════════
#  Cell 6 (★ NEW v2)  ·  Excel Export 모듈 import & 등록
# ═══════════════════════════════════════════════════════════════
#  ★ v2.2 자가검증:
#    Jupyter 는 한 번 import 한 모듈을 sys.modules 에 캐시하므로,
#    DATA/ 폴더에 새 모듈 파일을 덮어써도 같은 세션에서는 옛 코드가
#    계속 사용됩니다. 이 셀은:
#      (1) sys.modules 에서 옛 모듈 제거 → 강제 리로드
#      (2) __version__ 검증 → 옵션 B 적용 모듈인지 확인
#    실행 후 마지막 줄에 "✓ Module version: v2.2-optionB" 가 보이면 정상.
# ═══════════════════════════════════════════════════════════════
import sys, importlib

# (1) 강제 리로드 — 옛 import 캐시 제거
for mod_name in list(sys.modules.keys()):
    if "FCFF_Result_export_excel_module" in mod_name:
        del sys.modules[mod_name]

# (2) Fresh import
import DATA.FCFF_Result_export_excel_module as _fxm
importlib.reload(_fxm)

# (3) 버전 자가검증
EXPECTED_VERSION = "v2.3-mainsheet-fix"
actual_version = getattr(_fxm, "__version__", "<not set>")
if actual_version != EXPECTED_VERSION:
    raise RuntimeError(
        f"\n[ERROR] Excel Export 모듈 버전 불일치!\n"
        f"  expected: {EXPECTED_VERSION}\n"
        f"  actual:   {actual_version}\n"
        f"  → DATA/FCFF_Result_export_excel_module.py 가 최신 파일로 덮어쓰여 있는지 확인하세요.\n"
        f"  → 파일 위치: {getattr(_fxm, '__file__', '<unknown>')}"
    )

# (4) 노출되는 공개 함수
configure_excel_export = _fxm.configure
register_excel_export  = _fxm.register_excel_export

# (5) 한국 모형의 상수·로거 전달
configure_excel_export(
    debt_keys=DEBT_KEYS,                                # Cell 2 정의
    cash_keys=CASH_KEYS,
    log_func=log,                                       # 한국 v2 내부 logger
    marketcap_table="ks_listed_company_daily_marketcap",
    blume_w_raw=0.67,
    blume_w_market=0.33,
)

# (6) KoreaDCFModel 클래스에 메서드 attach
register_excel_export(KoreaDCFModel)

print(f"\n✓ Module version: {actual_version}")
print(f"  Module file:    {_fxm.__file__}")
print(f"  → Net Debt(B40), WACC(B50) 가 model.valuation 으로 고정됨 → DCF Δ = 0 보장")
print(f"  → Main 시트 forecast (R43-R50) 도 model.result_df 직접 입력 → Phase2 와 일치")

# 이후 사용:
#   model = KoreaDCFModel(ticker="A005930", engine=engine, ...).run()
#   out_path = model.export_korea_excel(out_dir="C:/reports")


[OK] KoreaDCFModel patched via FCFF_Result_export_excel_module:
     ① _fetch_korea_extras   (10Y monthly close: 종목 + KOSPI)
     ② export_korea_excel    (9-sheet workbook)

✓ Module version: v2.3-mainsheet-fix
  Module file:    C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\FCFF_Result_export_excel_module.py
  → Net Debt(B40), WACC(B50) 가 model.valuation 으로 고정됨 → DCF Δ = 0 보장
  → Main 시트 forecast (R43-R50) 도 model.result_df 직접 입력 → Phase2 와 일치


## Cell 8.5 · ★ NWC_Detail 시트 패치 (v1.2 신규 — US v16_2 이식)
미국 v16_2 의 `_build_nwc_detail` 을 한국 모형에 이식: 운전자본 구성(섹션 B) + **매출TTM / COGS TTM / DSO / DIO / DPO / CCC** 회전일수 진단(섹션 B2, 최근 24Q) + 최근 8Q 안정성 요약(CV → 안정/보통/불안정).

- 공유 모듈(`FCFF_Result_export_excel_module`, v2.3 버전검증)은 **수정하지 않음** — `export_korea_excel` 을 래핑해 저장된 Excel 에 시트를 추가 (9-sheet → 10-sheet)
- COGS(매출원가): `_fs_wide` alias 스캔 → 실패 시 `korea_fs_data_from_DG` DB fallback (`indicator LIKE '%매출원가%'`, 천원→원 환산) → 그래도 없으면 DIO/DPO 공란 + 경고
- 표시 단위 기본 **억원** (`NWC_DETAIL_UNIT_DIVISOR = 1e8`, 원 단위는 `1` 로 변경)
- 시트 생성 실패 시에도 기존 9-sheet Excel 은 정상 저장 (fail-safe)

In [21]:
# ═══════════════════════════════════════════════════════════════
#  Cell 8.5 (★ NEW v1.2)  ·  NWC_Detail 시트 패치 — US v16_2 이식
# ═══════════════════════════════════════════════════════════════
#  미국 FCFF_DCF_Individual_v16_2 의 NWC_Detail 시트(운전자본 구성 +
#  DSO/DIO/DPO/CCC 회전일수 진단)를 한국 모형에 이식합니다.
#
#  구조 (US v16_2 와 동일):
#    · 섹션 A  : γ 적합 요약 (γ, 방법, R², 표본수, 마지막 실측 NWC, NWC 산출방식)
#    · 섹션 B  : 실측 운전자본 구성 (매출채권·재고·기타 / 매입채무·기타) 최근 24Q
#    · 섹션 B2 : 회전일수 진단 — 매출TTM·COGS TTM·DSO·DIO·DPO·CCC (최근 24Q)
#                + 최근 8Q 안정성 요약 (평균/표준편차/CV → 안정·보통·불안정)
#    · 섹션 C  : 해석 가이드
#
#  한국 특화 처리:
#    ① COGS(매출원가): _fs_wide 컬럼 alias 스캔 → 없으면 TABLE_FS
#       (korea_fs_data_from_DG) 에서 indicator LIKE '%매출원가%' 직접 조회
#       (천원→원 환산). 그래도 없으면 DIO/DPO 공란 + 경고 (DSO 는 정상).
#    ② NWC 정의: 모형이 실제 사용한 방식(operating_v7 / legacy_proxy)을
#       그대로 재현 → 섹션 B 집계 NWC = γ 적합에 쓰인 NWC 와 정확히 일치.
#    ③ 표시 단위: NWC_DETAIL_UNIT_DIVISOR (기본 1e8 = 억원). 원 단위로
#       보려면 1 로 변경.
#
#  적용 방식: 공유 모듈(FCFF_Result_export_excel_module, v2.3 버전검증)은
#  건드리지 않고, export_korea_excel 을 래핑 → 모듈이 저장한 9-sheet Excel
#  을 다시 열어 NWC_Detail 시트를 추가 저장 (총 10-sheet). 시트 생성이
#  실패해도 기존 9-sheet Excel 은 정상 저장됩니다.
# ═══════════════════════════════════════════════════════════════
from typing import Optional, Tuple

import numpy as np
import pandas as pd
from sqlalchemy import text as _sql_text
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── 표시 단위 (★ 필요 시 여기만 수정) ───────────────────────────
NWC_DETAIL_UNIT_DIVISOR = 1e8      # 1e8=억원 · 1e12=조원 · 1=원
NWC_DETAIL_UNIT_LABEL   = "억원"    # 헤더에 표기할 단위명

# ── COGS(매출원가) wide 컬럼 alias 후보 ─────────────────────────
KOREA_COGS_ALIASES = [
    "cogs", "cost_of_goods_sold", "cost_of_sales", "cost_of_revenue",
    "매출원가",
]

# ── 시트 스타일 상수 (US v16_2 팔레트와 동일 계열) ──────────────
_K_NAVY, _K_TEAL, _K_GOLD = "203864", "2E75B6", "BF8F00"
_K_ZEBRA, _K_LINK_BG      = "F2F2F2", "FFF2CC"
_K_THIN   = Side(style="thin", color="BFBFBF")
_K_BORDER = Border(left=_K_THIN, right=_K_THIN, top=_K_THIN, bottom=_K_THIN)
_K_FMT_INT, _K_FMT_DEC, _K_FMT_DAYS = "#,##0", "0.0000", "0.0"


def _k_section_header(ws, row, title, span=13, color=_K_TEAL):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=span)
    c = ws.cell(row=row, column=1, value=title)
    c.font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    c.fill = PatternFill("solid", start_color=color)
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20
    return row + 1


def _k_num_series(wide: pd.DataFrame, col: Optional[str]) -> pd.Series:
    """wide 컬럼 → 숫자 Series (없으면 0)."""
    if col is not None and col in wide.columns:
        return pd.to_numeric(wide[col], errors="coerce").fillna(0.0)
    return pd.Series(0.0, index=wide.index)


def _korea_nwc_components(model) -> Tuple[pd.DataFrame, str]:
    """
    모형이 γ 적합에 실제 사용한 NWC 정의(operating_v7 / legacy_proxy)를
    그대로 재현한 구성요소 시계열.
    반환 DataFrame 컬럼:
      date, receivables, inventories, other_ca, op_ca,
      payables, other_cl, op_cl, nwc, cash, st_debt
    (other_* 는 합계 − 매출채권/재고 · − 매입채무 잔차 → 합이 집계와 정확히 일치)
    """
    wide = model._fs_wide
    method = getattr(model, "_nwc_method", None)
    if method is None:                       # run() 전 호출 등 방어
        model._compute_bs_nwc_series()
        method = getattr(model, "_nwc_method", "legacy_proxy")

    rec = _k_num_series(wide, model._resolve_alias("receivables"))
    inv = _k_num_series(wide, model._resolve_alias("inventories"))

    if method == "operating_v7":
        op_ca, _ = model._sum_components(model.NWC_OP_ASSET_ALIASES)
        op_cl, _ = model._sum_components(model.NWC_OP_LIAB_ALIASES)
        pay = _k_num_series(wide, model._resolve_from(model.NWC_OP_LIAB_ALIASES,
                                                      "payables"))
    else:                                    # legacy_proxy: (Recv+Inv) − 단기성 재무부채
        op_ca = rec + inv
        st = pd.Series(0.0, index=wide.index)
        for k in ("short_term_debt", "current_lt_debt", "lease_liab"):
            st = st + _k_num_series(wide, model._resolve_alias(k))
        op_cl = st
        pay = pd.Series(0.0, index=wide.index)

    cash_col = next((k for k in CASH_KEYS if k in wide.columns), None)
    cash    = _k_num_series(wide, cash_col)
    st_debt = _k_num_series(wide, model._resolve_alias("short_term_debt"))

    df = pd.DataFrame({
        "date":        wide.index,
        "receivables": rec.values,
        "inventories": inv.values,
        "op_ca":       op_ca.values,
        "payables":    pay.values,
        "op_cl":       op_cl.values,
        "cash":        cash.values,
        "st_debt":     st_debt.values,
    }).reset_index(drop=True)
    df["other_ca"] = df["op_ca"] - df["receivables"] - df["inventories"]
    df["other_cl"] = df["op_cl"] - df["payables"]
    df["nwc"]      = df["op_ca"] - df["op_cl"]
    return df, method


def _korea_load_cogs(model) -> Tuple[pd.Series, str]:
    """
    COGS(매출원가) 분기 시계열 (단위: 원, _fs_wide 와 동일).
      1) _fs_wide 컬럼 alias 스캔
      2) 실패 시 TABLE_FS 롱테이블에서 indicator LIKE '%매출원가%' 직접 조회
         (관측치 최다 indicator 채택, 천원→원 환산)
    반환: (Series[date-index], source 설명)  — 실패 시 (빈 Series, 'missing')
    """
    wide = model._fs_wide
    for cand in KOREA_COGS_ALIASES:
        if cand in wide.columns:
            s = pd.to_numeric(wide[cand], errors="coerce").abs()
            if s.notna().sum() >= 4:
                return s, f"wide:{cand}"

    # DB fallback — korea_fs_data_from_DG 롱포맷 직접 조회
    try:
        q = _sql_text(
            f"SELECT date, indicator, value FROM {TABLE_FS} "
            f"WHERE ticker = :tk AND indicator LIKE :pat"
        )
        with model.engine.connect() as conn:
            raw = pd.read_sql(q, conn,
                              params={"tk": model.ticker_dg, "pat": "%매출원가%"})
        if raw.empty:
            return pd.Series(dtype=float), "missing"
        raw["date"]  = pd.to_datetime(raw["date"])
        raw["value"] = pd.to_numeric(raw["value"], errors="coerce")
        best = (raw.groupby("indicator")["value"]
                   .apply(lambda s: int(s.notna().sum()))
                   .sort_values(ascending=False))
        ind = best.index[0]
        sub = (raw[raw["indicator"] == ind]
               .dropna(subset=["value"])
               .drop_duplicates(subset=["date"], keep="last")
               .set_index("date")["value"]
               .sort_index())
        sub = sub.abs() * FS_UNIT_MULTIPLIER          # 천원 → 원
        sub = sub.reindex(wide.index)                 # 재무제표 분기 격자에 정렬
        if sub.notna().sum() >= 4:
            return sub, f"db:{ind} (n={best.iloc[0]})"
        return pd.Series(dtype=float), "missing"
    except Exception as e:                            # DB 실패해도 시트는 생성
        log(model.ticker_dg, f"⚠️  COGS DB fallback 실패: {e}")
        return pd.Series(dtype=float), "missing"


def _build_korea_nwc_detail(wb, model, ticker):
    """NWC_Detail 시트 (US v16_2 _build_nwc_detail 한국판)."""
    if "NWC_Detail" in wb.sheetnames:                 # 재실행 시 중복 방지
        del wb["NWC_Detail"]
    ws = wb.create_sheet("NWC_Detail")
    ws.sheet_view.showGridLines = False

    U = float(NWC_DETAIL_UNIT_DIVISOR)
    UL = NWC_DETAIL_UNIT_LABEL if U != 1 else "원"

    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=13)
    t = ws.cell(row=1, column=1,
                value=f"{ticker} — 운전자본(NWC) 구성 & 회전일수 진단  "
                      f"[v1.2 · US v16_2 이식 · 단위: {UL}]")
    t.font = Font(name="Calibri", size=14, bold=True, color="FFFFFF")
    t.fill = PatternFill("solid", start_color=_K_NAVY)
    t.alignment = Alignment(horizontal="center", vertical="center")
    ws.row_dimensions[1].height = 26

    comp, nwc_method = _korea_nwc_components(model)

    # 분기 매출 (원) — NWC/Sales 및 TTM 계산용
    rev = pd.to_numeric(model._fs_wide["revenue"], errors="coerce")
    rev_map = dict(zip(model._fs_wide.index, rev.values))

    # ── 섹션 A: γ 적합 요약 ─────────────────────────────────────
    r = _k_section_header(ws, 3, "A.  NWC 예측계수 γ  (예측 NWC = γ × Sales)",
                          color=_K_TEAL)
    gamma = getattr(model, "_nwc_to_sales", None)
    if gamma is None:
        gamma, _m = model.estimate_nwc_coef()

    # R²·표본수 재계산 (모형 γ 적합과 동일한 merge 로직)
    nwc_fit = comp[["date", "nwc"]].copy()
    sales_df = pd.DataFrame({"date": model._fs_wide.index, "rev": rev.values})
    merged = sales_df.merge(nwc_fit, on="date", how="inner").dropna()
    merged = merged[merged["rev"] > 0]
    if len(merged) >= 2:
        slope, r2, nsamp = model._ols_ratio(merged["rev"], merged["nwc"])
        fit_method = ("ols" if (not np.isnan(slope) and r2 >= OLS_MIN_R2
                                and nsamp >= OLS_MIN_SAMPLES) else "median_ratio")
    else:
        r2, nsamp, fit_method = float("nan"), len(merged), "n/a"

    last_nwc = getattr(model, "_last_actual_nwc", float("nan"))
    a_rows = [
        ("γ  (NWC/Sales)",  gamma,           _K_FMT_DEC),
        ("적합 방법",        fit_method,      None),
        (f"R²  (OLS 채택 기준 ≥ {OLS_MIN_R2:.2f})", r2, _K_FMT_DEC),
        ("표본 분기 수",     nsamp,           "#,##0"),
        (f"마지막 실측 집계 NWC ({UL})",
         (last_nwc / U if pd.notna(last_nwc) else float("nan")), _K_FMT_INT),
        ("NWC 산출 방식",
         ("v7 영업운전자본 (Damodaran)" if nwc_method == "operating_v7"
          else "legacy proxy (Recv+Inv − 단기재무부채)"), None),
    ]
    for label, val, fmt in a_rows:
        lc = ws.cell(row=r, column=1, value=label)
        lc.font = Font(name="Calibri", size=10)
        c = ws.cell(row=r, column=3, value=val)
        if fmt:
            c.number_format = fmt
        c.font = Font(name="Calibri", size=10, bold=True)
        c.alignment = Alignment(horizontal="right")
        c.border = _K_BORDER
        c.fill = PatternFill("solid", start_color=_K_LINK_BG)
        r += 1
    ws.cell(row=r, column=1,
            value="  ※ 예측(Forecast) NWC 는 γ 하나로 NWC=γ×Sales 로 산출됩니다. "
                  "따라서 예측 ΔNWC = γ×ΔSales 이며 구성요소 분해가 존재하지 않습니다.").font = \
        Font(italic=True, size=9, color="C0392B")
    r += 2

    # ── 섹션 B: 실측 운전자본 구성 (최근 24Q) ───────────────────
    r = _k_section_header(
        ws, r, f"B.  실측 재무제표 기반 운전자본 구성 (최근 24Q · 단위: {UL})",
        color=_K_TEAL)
    headers = ["Quarter", "매출채권", "재고자산", "기타영업유동자산", "영업유동자산",
               "매입채무", "기타영업유동부채", "영업유동부채", "집계 NWC",
               "ΔNWC", "NWC/Sales", "현금", "단기차입금"]
    for j, h in enumerate(headers, 1):
        c = ws.cell(row=r, column=j, value=h)
        c.font = Font(bold=True, color="FFFFFF", size=9)
        c.fill = PatternFill("solid", start_color=_K_TEAL)
        c.alignment = Alignment(horizontal="center", wrap_text=True)
        c.border = _K_BORDER
    r += 1

    detail_start = r
    if not comp.empty:
        sub = comp.tail(24).reset_index(drop=True)
        prev_nwc = None
        for _, row_ in sub.iterrows():
            dt = row_["date"]
            q = f"{dt.year}Q{dt.quarter}" if hasattr(dt, "year") else str(dt)
            nwc_v = float(row_["nwc"])
            dnwc = (nwc_v - prev_nwc) if prev_nwc is not None else float("nan")
            prev_nwc = nwc_v
            rev_v = rev_map.get(dt, float("nan"))
            ratio = (nwc_v / rev_v) if (pd.notna(rev_v) and rev_v) else float("nan")
            vals = [q,
                    row_["receivables"] / U, row_["inventories"] / U,
                    row_["other_ca"] / U,    row_["op_ca"] / U,
                    row_["payables"] / U,    row_["other_cl"] / U,
                    row_["op_cl"] / U,       nwc_v / U,
                    (dnwc / U if pd.notna(dnwc) else float("nan")), ratio,
                    row_["cash"] / U,        row_["st_debt"] / U]
            zebra = (PatternFill("solid", start_color=_K_ZEBRA)
                     if (r - detail_start) % 2 == 0 else None)
            for j, v in enumerate(vals, 1):
                is_nan = isinstance(v, float) and pd.isna(v)
                cc = ws.cell(row=r, column=j, value=(None if is_nan else v))
                cc.border = _K_BORDER
                cc.font = Font(name="Calibri", size=9)
                if zebra:
                    cc.fill = zebra
                if j == 1:
                    cc.alignment = Alignment(horizontal="center")
                elif j == 11:
                    cc.number_format = _K_FMT_DEC
                    cc.alignment = Alignment(horizontal="right")
                else:
                    cc.number_format = _K_FMT_INT
                    cc.alignment = Alignment(horizontal="right")
            r += 1
    else:
        ws.cell(row=r, column=1,
                value="(재무상태표 구성요소를 확보하지 못했습니다 — BS line item 누락)").font = \
            Font(italic=True, color="595959")
        r += 1
    if nwc_method != "operating_v7":
        ws.cell(row=r, column=1,
                value="  ⚠ legacy proxy 모드: '영업유동부채' 열은 단기재무부채"
                      "(단기차입금+유동성장기부채+리스) 합계입니다 (v7 영업부채 항목 DB 미적재).").font = \
            Font(italic=True, size=9, color="C0392B")
        r += 1
    r += 1

    # ── 섹션 B2: 회전일수(DSO/DIO/DPO) 진단 — TTM 기준 ──────────
    r = _k_section_header(
        ws, r, "B2.  회전일수(DSO/DIO/DPO) 진단 — TTM 기준", color=_K_TEAL)

    cogs, cogs_src = _korea_load_cogs(model)
    inc = pd.DataFrame({"date": model._fs_wide.index,
                        "revenue": rev.values}).sort_values("date")
    inc["_cogs"] = (cogs.reindex(model._fs_wide.index).values
                    if not cogs.empty else np.nan)
    inc = inc.reset_index(drop=True)
    inc["rev_ttm"]  = inc["revenue"].rolling(4).sum()
    inc["cogs_ttm"] = pd.to_numeric(inc["_cogs"], errors="coerce").rolling(4).sum()

    if not comp.empty:
        turn = comp[["date", "receivables", "inventories", "payables"]].merge(
            inc[["date", "rev_ttm", "cogs_ttm"]], on="date", how="left")
        turn["DSO"] = turn["receivables"] / turn["rev_ttm"]  * 365.0
        turn["DIO"] = turn["inventories"] / turn["cogs_ttm"] * 365.0
        turn["DPO"] = turn["payables"]    / turn["cogs_ttm"] * 365.0
        if nwc_method != "operating_v7":              # legacy: 매입채무 없음 → DPO 무의미
            turn["DPO"] = np.nan
        turn["CCC"] = turn["DSO"] + turn["DIO"] - turn["DPO"].fillna(0)
        turn = turn.replace([np.inf, -np.inf], np.nan)
    else:
        turn = pd.DataFrame()

    t_headers = ["Quarter", f"매출 TTM ({UL})", f"COGS TTM ({UL})",
                 "DSO(일)", "DIO(일)", "DPO(일)", "CCC(일)"]
    for j, h in enumerate(t_headers, 1):
        c = ws.cell(row=r, column=j, value=h)
        c.font = Font(bold=True, color="FFFFFF", size=9)
        c.fill = PatternFill("solid", start_color=_K_TEAL)
        c.alignment = Alignment(horizontal="center", wrap_text=True)
        c.border = _K_BORDER
    r += 1

    t_start = r
    if not turn.empty:
        sub_t = turn.tail(24).reset_index(drop=True)
        for _, row_ in sub_t.iterrows():
            dt = row_["date"]
            q = f"{dt.year}Q{dt.quarter}" if hasattr(dt, "year") else str(dt)
            vals = [q,
                    (row_["rev_ttm"]  / U if pd.notna(row_["rev_ttm"])  else float("nan")),
                    (row_["cogs_ttm"] / U if pd.notna(row_["cogs_ttm"]) else float("nan")),
                    row_.get("DSO"), row_.get("DIO"), row_.get("DPO"), row_.get("CCC")]
            zebra = (PatternFill("solid", start_color=_K_ZEBRA)
                     if (r - t_start) % 2 == 0 else None)
            for j, v in enumerate(vals, 1):
                is_nan = isinstance(v, float) and pd.isna(v)
                cc = ws.cell(row=r, column=j, value=(None if is_nan else v))
                cc.border = _K_BORDER
                cc.font = Font(name="Calibri", size=9)
                if zebra:
                    cc.fill = zebra
                if j == 1:
                    cc.alignment = Alignment(horizontal="center")
                elif j in (2, 3):
                    cc.number_format = _K_FMT_INT
                    cc.alignment = Alignment(horizontal="right")
                else:
                    cc.number_format = _K_FMT_DAYS
                    cc.alignment = Alignment(horizontal="right")
            r += 1

        # ── B2-요약: 최근 8Q 안정성 (CV) ────────────────────────
        recent = turn.tail(8)
        r += 1
        r = _k_section_header(
            ws, r, "B2-요약.  최근 8Q 회전일수 안정성 (CV 낮을수록 안정 → 예측 신뢰↑)",
            color=_K_GOLD)
        sm_hdr = ["지표", "최근값", "8Q 평균", "8Q 표준편차", "변동계수 CV", "안정성"]
        for j, h in enumerate(sm_hdr, 1):
            c = ws.cell(row=r, column=j, value=h)
            c.font = Font(bold=True, color="FFFFFF", size=9)
            c.fill = PatternFill("solid", start_color=_K_NAVY)
            c.alignment = Alignment(horizontal="center")
            c.border = _K_BORDER
        r += 1
        for metric in ["DSO", "DIO", "DPO", "CCC"]:
            s = recent[metric].replace([np.inf, -np.inf], np.nan).dropna()
            if s.empty:
                continue
            last = float(s.iloc[-1]); mean = float(s.mean()); std = float(s.std())
            cv = (std / abs(mean)) if abs(mean) > 1e-9 else float("nan")
            stab = ("안정" if (pd.notna(cv) and cv < 0.10)
                    else "보통" if (pd.notna(cv) and cv < 0.25) else "불안정")
            vals = [metric, last, mean, std, cv, stab]
            for j, v in enumerate(vals, 1):
                is_nan = isinstance(v, float) and pd.isna(v)
                cc = ws.cell(row=r, column=j, value=(None if is_nan else v))
                cc.border = _K_BORDER
                cc.font = Font(name="Calibri", size=9, bold=(j == 6))
                if j == 1:
                    cc.alignment = Alignment(horizontal="center")
                elif j == 5:
                    cc.number_format = "0.0%"
                    cc.alignment = Alignment(horizontal="right")
                elif j == 6:
                    cc.alignment = Alignment(horizontal="center")
                    _bg = {"안정": "C8E6C9", "보통": "FFF9C4",
                           "불안정": "FFCDD2"}.get(v, "FFFFFF")
                    cc.fill = PatternFill("solid", start_color=_bg)
                else:
                    cc.number_format = _K_FMT_DAYS
                    cc.alignment = Alignment(horizontal="right")
            r += 1
    else:
        ws.cell(row=r, column=1,
                value="(회전일수 산출 불가 — 구성요소 누락)").font = \
            Font(italic=True, color="595959")
        r += 1
    if cogs_src == "missing":
        ws.cell(row=r, column=1,
                value="  ⚠ COGS(매출원가)를 wide 컬럼·DB 어디서도 찾지 못해 "
                      "DIO/DPO 는 공란입니다. (DSO 는 매출 기반이라 정상 산출)").font = \
            Font(italic=True, size=9, color="C0392B")
        r += 1
    else:
        ws.cell(row=r, column=1,
                value=f"  · COGS source: {cogs_src}").font = \
            Font(italic=True, size=9, color="595959")
        r += 1
    r += 1

    # ── 섹션 C: 해석 가이드 ─────────────────────────────────────
    r = _k_section_header(ws, r, "C.  해석 가이드", color=_K_GOLD)
    notes = [
        f"• 표시 단위: {UL} (재무제표 천원 값을 원 환산 후 {NWC_DETAIL_UNIT_DIVISOR:,.0f} 로 나눔). "
        "원 단위로 보려면 패치 셀의 NWC_DETAIL_UNIT_DIVISOR = 1 로 변경.",
        "• 집계 NWC = 영업유동자산 − 영업유동부채 "
        "(v7: (매출채권+재고+선급비용) − (매입채무+미지급비용+미지급금+선수금+계약부채)). "
        "'기타' 열은 잔차로, 합이 집계와 정확히 일치합니다.",
        "• 섹션 B(실측)의 각 항목 증감으로 실제 ΔNWC 가 매출채권 증가/재고 축적/매입채무 변화 중 "
        "무엇에서 왔는지 귀속할 수 있습니다.",
        "• 예측 ΔNWC(=γ×ΔSales)는 이 분해가 불가능합니다. 실제 원인 규명은 섹션 B 실측 line item 으로만 가능합니다.",
        "• NWC/Sales 비율이 시기에 따라 크게 변한다면 단일 γ 모델의 예측력이 약하다는 신호 → "
        "구성요소별 예측(Tier2) 검토 대상.",
        "• [B2] 회전일수 = 분기말 잔액 ÷ TTM 유량 × 365. "
        "DSO=매출채권/매출TTM, DIO=재고/COGS_TTM, DPO=매입채무/COGS_TTM, CCC=DSO+DIO−DPO.",
        "• B2-요약의 변동계수(CV)가 낮으면(안정) 회전일수가 일정 → DSO/DIO/DPO 기반 Tier2 예측이 "
        "정확·안정적. CV가 높으면 회전일수 자체가 추세/충격을 타므로 별도 정규화 필요.",
        "• 실무 판단: 회전일수가 안정적인데 단일 γ(NWC/Sales) 오차가 크다면 → Tier2(회전일수) 도입 "
        "이득이 큼. 회전일수마저 불안정하면 raw 실측도 예측 부적합.",
    ]
    for n in notes:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=13)
        c = ws.cell(row=r, column=1, value=n)
        c.font = Font(name="Calibri", size=9, color="333333")
        c.alignment = Alignment(horizontal="left", vertical="center", wrap_text=True)
        ws.row_dimensions[r].height = 26
        r += 1

    widths = [10, 13, 13, 15, 14, 13, 15, 14, 13, 13, 11, 12, 12]
    for j, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(j)].width = w
    return ws


# ═══════════════════════════════════════════════════════════════
#  export_korea_excel 래핑 — 9-sheet 저장 후 NWC_Detail 추가 (10-sheet)
# ═══════════════════════════════════════════════════════════════
_curr_export = KoreaDCFModel.export_korea_excel
if not getattr(_curr_export, "__nwc_detail_wrapper__", False):
    KoreaDCFModel._export_korea_excel_base = _curr_export   # 원본 보존


def _export_korea_excel_with_nwc_detail(self, out_dir, **kwargs):
    out_path = self._export_korea_excel_base(out_dir=out_dir, **kwargs)
    try:
        wb = load_workbook(out_path)
        _build_korea_nwc_detail(wb, self, self.ticker_dg)
        wb.save(out_path)
        if getattr(self, "verbose", False):
            log(self.ticker_dg, "NWC_Detail 시트 추가 완료 (10-sheet)")
    except Exception as e:                      # 시트 실패 ≠ export 실패
        log(self.ticker_dg, f"⚠️  NWC_Detail 시트 생성 실패 (9-sheet 유지): {e}")
        import traceback as _tb
        _tb.print_exc()
    return out_path


_export_korea_excel_with_nwc_detail.__nwc_detail_wrapper__ = True
KoreaDCFModel.export_korea_excel = _export_korea_excel_with_nwc_detail

print("✓ NWC_Detail 패치 등록 완료")
print(f"  · export_korea_excel → 9-sheet 저장 후 NWC_Detail 시트 추가 (총 10-sheet)")
print(f"  · 표시 단위: {NWC_DETAIL_UNIT_LABEL} (divisor={NWC_DETAIL_UNIT_DIVISOR:,.0f})")
print(f"  · COGS 소스: wide alias {KOREA_COGS_ALIASES} → 실패 시 {TABLE_FS} DB fallback")


✓ NWC_Detail 패치 등록 완료
  · export_korea_excel → 9-sheet 저장 후 NWC_Detail 시트 추가 (총 10-sheet)
  · 표시 단위: 억원 (divisor=100,000,000)
  · COGS 소스: wide alias ['cogs', 'cost_of_goods_sold', 'cost_of_sales', 'cost_of_revenue', '매출원가'] → 실패 시 korea_fs_data_from_DG DB fallback


## Cell 8.6 · ★ OPM_Model 시트 + Phase2 미드사이클 동기화 (v1.5 신규)

회귀 OPM(α·β·R²·홀드아웃 MAE), 분기 OPM 실적/예측과 구 TS 방식 비교, Phase 2 미드사이클 수렴 경로, Excel TP 정합성(Δ)을 담은 `OPM_Model` 시트를 추가하고, `Phase2_LongTerm` 의 Phase 2/3 FCFF 를 미드사이클 반영값으로 동기화합니다.

> Cell 7·8·8.5 를 다시 실행했다면 이 셀도 반드시 재실행하세요.

In [22]:
# ═══════════════════════════════════════════════════════════════
#  Cell 8.6 (★ NEW v1.5)  ·  OPM_Model 시트 + Phase2 미드사이클 동기화 패치
# ═══════════════════════════════════════════════════════════════
#  공유 모듈(FCFF_Result_export_excel_module v2.3)은 수정하지 않고,
#  export_korea_excel 을 한 번 더 래핑해 저장된 Excel 을 후처리합니다.
#    (1) Phase2_LongTerm : Phase 2 FCFF 를 model.valuation['ph2_annual'](미드사이클 반영)
#        으로 덮어쓰고 Phase 3 를 g_term 으로 재계산 → DCF_Valuation TP = model TP 유지
#    (2) OPM_Model 시트 신규 : 회귀식(α·β·R²·홀드아웃 MAE) · 분기 OPM 실적/예측 ·
#        TS 방식 비교 · 미드사이클 수렴 경로 · Excel TP 정합성 검증
#  Cell 8.5(NWC_Detail) 다음에 실행 — 스타일 헬퍼(_k_section_header 등)를 재사용합니다.
# ═══════════════════════════════════════════════════════════════
OPM_SHEET_UNIT_DIVISOR = 1e8     # 표시 단위: 1e8=억원
OPM_SHEET_UNIT_LABEL   = "억원"
OPM_COMPARE_TS         = True    # True → 구 v1.4 TS 방식 OPM 을 비교열로 함께 표시 (3개 모형 추가 적합)


def _opm_style_row(ws, r, ncol, header=False):
    for j in range(1, ncol + 1):
        c = ws.cell(row=r, column=j)
        c.border = _K_BORDER
        if header:
            c.font = Font(name="Calibri", size=10, bold=True, color="FFFFFF")
            c.fill = PatternFill("solid", start_color=_K_NAVY)
            c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)


def _sync_phase2_sheet(wb, model) -> Dict:
    """Phase2_LongTerm 의 Phase 2/3 FCFF 를 model 값으로 동기화하고 Excel TP 를 재현 검증."""
    v = model.valuation
    ws = wb["Phase2_LongTerm"]
    ph2 = list(v["ph2_annual"]); g_term = float(v["g_terminal"])
    mc = v.get("midcycle", {}) or {}
    rows = [r for r in range(14, ws.max_row + 1) if ws.cell(row=r, column=3).value in
            ("Phase 1", "Phase 2", "Phase 3")]
    k2, prev = 0, None
    for r in rows:
        ph = ws.cell(row=r, column=3).value
        if ph == "Phase 1":
            prev = float(ws.cell(row=r, column=5).value)
            continue
        if ph == "Phase 2" and k2 < len(ph2):
            val = float(ph2[k2])
            ws.cell(row=r, column=4, value=(val / prev - 1) if prev and prev > 0 else None)
            ws.cell(row=r, column=5, value=val)
            if mc.get("applied"):
                ws.cell(row=r, column=6,
                        value=f"AR(1) ρ={v['moat_rho']:.3f} + 미드사이클 OPM {mc['opm_path'][k2]:.1%}")
            k2 += 1
            prev = val
        elif ph == "Phase 3" and prev is not None:
            val = prev * (1 + g_term)
            ws.cell(row=r, column=4, value=g_term)
            ws.cell(row=r, column=5, value=val)
            prev = val
    if mc.get("applied"):
        ws.cell(row=10, column=1,
                value=(f"미드사이클 OPM 수렴(v1.5): {mc['opm_end']:.1%} → {mc['opm_mid']:.1%} "
                       f"({mc['mid_stat']}, {mc['mid_window']}), {mc['years']}년 선형 — 상세는 OPM_Model 시트")
                ).font = Font(italic=True, size=9, color="C0392B")

    # Excel DCF 재현: DCF_Valuation 은 Phase2_LongTerm E 열(최대 17행)을 할인 후 마지막 행에서 TV
    wacc = float(v["wacc"])
    fc = [float(ws.cell(row=r, column=5).value) for r in rows]
    n_dcf = sum(1 for r in range(14, 60) if wb["DCF_Valuation"].cell(row=r, column=4).value
                and str(wb["DCF_Valuation"].cell(row=r, column=4).value).startswith("=Phase2"))
    fc = fc[:n_dcf] if n_dcf else fc
    pv = sum(f / (1 + wacc) ** (t + 1) for t, f in enumerate(fc))
    tv = fc[-1] * (1 + g_term) / (wacc - g_term)
    if v.get("tv_capped") and fc[-1] > 0:
        pass    # TV 배수 상한 적용 종목은 Excel 수식 체인과 model 이 원래 다를 수 있음 (모듈 v2.3 동일)
    ev_x = pv + tv / (1 + wacc) ** len(fc)
    tp_x = (ev_x - float(v["net_debt"])) / float(v["shares"]) if v.get("shares") else np.nan
    return {"excel_tp": tp_x, "model_tp": float(v["target_price"]),
            "delta": tp_x - float(v["target_price"]), "n_dcf_years": len(fc)}


def _build_opm_model_sheet(wb, model, tp_check: Dict):
    if "OPM_Model" in wb.sheetnames:
        del wb["OPM_Model"]
    ws = wb.create_sheet("OPM_Model")
    v, info = model.valuation, dict(model.valuation.get("opm_info", {}))
    mc = v.get("midcycle", {}) or {}
    U, UL = OPM_SHEET_UNIT_DIVISOR, OPM_SHEET_UNIT_LABEL
    ws.cell(row=1, column=1, value=f"{model.ticker_dg} — OPM 추정 모형 (v1.5) · 미드사이클 수렴").font = \
        Font(name="Calibri", size=13, bold=True, color=_K_NAVY)
    r = 3

    # ── A. 추정 요약 ─────────────────────────────────────────────
    r = _k_section_header(ws, r, "A.  OPM 추정 요약", span=9)
    m_used = info.get("mode_used", "?")
    items = [
        ("OPM_MODE (요청 → 사용)", f"{info.get('mode_requested')} → {m_used}"),
        ("회귀 방식", info.get("reg_method", "—")),
        ("회귀 창", f"{info.get('window', '—')} ({info.get('window_q', '—')}Q)"),
        ("α (level/diff: 원 · log: 절편)", info.get("reg_alpha")),
        ("β (공헌이익률 / 탄력성)", info.get("reg_beta")),
        ("R²", info.get("reg_r2")),
        ("표본 수", info.get("reg_n")),
        ("홀드아웃 MAE (조원, 방식별)",
         ", ".join(f"{k}={val/1e12:.3f}" for k, val in (info.get("holdout_mae") or {}).items()) or "—"),
        ("OPM 허용범위 (clip)", f"[{info.get('clip_lo', np.nan):.1%}, {info.get('clip_hi', np.nan):.1%}]"),
        ("clip 적용 분기 수", info.get("n_clipped", "—")),
        ("fallback 사유", info.get("fallback_reason") or info.get("ts_note") or "—"),
    ]
    for k, val in items:
        ws.cell(row=r, column=1, value=k).font = Font(bold=True, size=10)
        c = ws.cell(row=r, column=3, value=val)
        if isinstance(val, float):
            c.number_format = "#,##0" if abs(val) > 1e3 else "0.0000"
        r += 1
    if info.get("reg_method") == "level" and info.get("reg_alpha") is not None:
        a, b = info["reg_alpha"], info["reg_beta"]
        ws.cell(row=r, column=1,
                value=(f"해석: 분기 고정비 ≈ {-a/1e12:,.2f}조원, 매출 1원 증가 시 영업이익 {b:.3f}원 증가 "
                       f"→ OPM = β + α/매출 은 매출이 커질수록 β({b:.1%})에 수렴")).font = \
            Font(italic=True, size=9, color="595959")
        r += 1
    r += 1

    # ── B. 분기 OPM 실적 / 예측 ──────────────────────────────────
    r = _k_section_header(ws, r, f"B.  분기 매출·영업이익·OPM (단위: {UL})", span=9)
    hdr = ["분기", "구분", "매출", "영업이익", "OPM(적용)", "회귀 OPM(clip 전)",
           "TS 방식 OPM(v1.4 비교)", "OPM 차이(적용−TS)", "비고"]
    for j, h_ in enumerate(hdr, 1):
        ws.cell(row=r, column=j, value=h_)
    _opm_style_row(ws, r, len(hdr), header=True); r += 1
    hist = model._opm_hist_frame().tail(12)
    for idx, row in hist.iterrows():
        vals = [str(row["per"]), "실적", row["revenue"] / U, row["operating_income"] / U, row["opm"]]
        for j, x in enumerate(vals, 1):
            ws.cell(row=r, column=j, value=x)
        _opm_style_row(ws, r, len(hdr)); r += 1
    df = model.result_df
    sales_fc = pd.Series(df["sales_forecast"].values, index=pd.to_datetime(df["date"]))
    ts_opm = None
    if OPM_COMPARE_TS:
        try:
            ts_opm = model._opm_timeseries(sales_fc, record=False).values
        except Exception:
            ts_opm = None
    raw = info.get("opm_raw") or [None] * len(df)
    for i, row in df.reset_index(drop=True).iterrows():
        t = None if ts_opm is None else float(ts_opm[i])
        vals = [row["quarter"], "예측", row["sales_forecast"] / U, row["ebit"] / U, row["opm_forecast"],
                raw[i] if i < len(raw) else None, t,
                (row["opm_forecast"] - t) if t is not None else None,
                "clip" if (raw[i] is not None and abs(raw[i] - row["opm_forecast"]) > 1e-9) else ""]
        for j, x in enumerate(vals, 1):
            ws.cell(row=r, column=j, value=x)
        _opm_style_row(ws, r, len(hdr))
        for j in range(1, len(hdr) + 1):
            ws.cell(row=r, column=j).fill = PatternFill("solid", start_color=_K_LINK_BG)
        r += 1
    for rr in range(r - len(df) - len(hist), r):
        for j in (3, 4):
            ws.cell(row=rr, column=j).number_format = "#,##0"
        for j in (5, 6, 7, 8):
            ws.cell(row=rr, column=j).number_format = "0.0%"
    r += 1

    # ── C. 미드사이클 수렴 ───────────────────────────────────────
    r = _k_section_header(ws, r, "C.  Phase 2 미드사이클 OPM 수렴", span=9, color=_K_GOLD)
    if not mc.get("applied"):
        ws.cell(row=r, column=1, value=f"미적용 — {mc.get('reason')}").font = Font(italic=True)
        r += 2
    else:
        for k, val in [("Phase 1 말 OPM (최근 4Q 가중)", mc["opm_end"]),
                       (f"미드사이클 OPM ({mc['mid_stat']}, {mc['mid_window']})", mc["opm_mid"]),
                       ("수렴 기간 (년, 선형)", mc["years"])]:
            ws.cell(row=r, column=1, value=k).font = Font(bold=True, size=10)
            c = ws.cell(row=r, column=3, value=val)
            c.number_format = "0.0%" if isinstance(val, float) else "0"
            r += 1
        r += 1
        hdr2 = ["Phase2 연차", "매출 경로", "OPM 경로", "FCFF(조정 전)", "OPM 조정액(세후)",
                "FCFF(조정 후)", "조정률"]
        for j, h_ in enumerate(hdr2, 1):
            ws.cell(row=r, column=j, value=h_)
        _opm_style_row(ws, r, len(hdr2), header=True); r += 1
        pre = mc["ph2_annual_pre"]; post = mc["ph2_annual"]
        for k in range(len(pre)):
            vals = [k + 1, mc["sales_path"][k] / U, mc["opm_path"][k], pre[k] / U,
                    mc["adjustment"][k] / U, post[k] / U,
                    (post[k] / pre[k] - 1) if pre[k] else None]
            for j, x in enumerate(vals, 1):
                c = ws.cell(row=r, column=j, value=x)
                c.number_format = {3: "0.0%", 7: "0.0%"}.get(j, "#,##0")
            _opm_style_row(ws, r, len(hdr2)); r += 1
        r += 1

    # ── D. 정합성 · 해석 ─────────────────────────────────────────
    r = _k_section_header(ws, r, "D.  Excel TP 정합성 · 해석 가이드", span=9, color=_K_TEAL)
    for k, val in [("model TP", tp_check["model_tp"]), ("Excel 재현 TP (Phase2 시트 기준)", tp_check["excel_tp"]),
                   ("Δ (Excel − model)", tp_check["delta"])]:
        ws.cell(row=r, column=1, value=k).font = Font(bold=True, size=10)
        ws.cell(row=r, column=3, value=val).number_format = "#,##0"
        r += 1
    notes = [
        "• 회귀 OPM 은 매출 예측과 연동됩니다: level 방식 OPM = β + α/매출 → 매출이 늘면 영업레버리지로 OPM 상승, β 로 수렴.",
        "• 구 v1.4 는 OPM 시계열만 따로 예측 후 [-30%, 50%] 고정 clip → 호황기 종목은 전 분기 50% 로 잘리는 문제가 있었습니다.",
        "• 회귀는 호황 구간 표본이 많으면 정점 마진을 연장할 수 있습니다 → Phase 2 에서 미드사이클 OPM 으로 수렴시켜 TV 과대평가를 막습니다.",
        "• 미드사이클 조정 = 매출 경로 × (OPM 경로 − Phase1 말 OPM) × (1−T). 매출 경로는 매출 g₀(yr2/yr1)를 FCFF 와 같은 ρ 로 g_term 까지 감쇠.",
        "• 매출 경로는 정점 매출을 유지한 채 마진만 정규화합니다. 가격 주도 사이클 업종은 매출도 함께 정상화될 수 있으므로 "
        "MIDCYCLE_STAT='weighted'(이익가중) 또는 MIDCYCLE_WINDOW_Q 조정으로 민감도를 확인하세요.",
    ]
    for n in notes:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=9)
        c = ws.cell(row=r, column=1, value=n)
        c.font = Font(size=9, color="333333")
        c.alignment = Alignment(wrap_text=True, vertical="center")
        ws.row_dimensions[r].height = 28
        r += 1
    for j, w in enumerate([16, 10, 14, 14, 12, 16, 18, 16, 8], 1):
        ws.column_dimensions[get_column_letter(j)].width = w
    return ws


_curr_export_86 = KoreaDCFModel.export_korea_excel
if not getattr(_curr_export_86, "__opm_model_wrapper__", False):
    KoreaDCFModel._export_korea_excel_pre_opm = _curr_export_86


def _export_korea_excel_with_opm(self, out_dir, **kwargs):
    out_path = self._export_korea_excel_pre_opm(out_dir=out_dir, **kwargs)
    try:
        wb = load_workbook(out_path)
        chk = _sync_phase2_sheet(wb, self)
        _build_opm_model_sheet(wb, self, chk)
        wb.save(out_path)
        log(self.ticker_dg, f"OPM_Model 시트 추가 · Phase2 동기화 완료 "
            f"(Excel TP Δ = {chk['delta']:,.0f}원)")
    except Exception as e:
        log(self.ticker_dg, f"[경고] OPM_Model/Phase2 동기화 실패 (기존 Excel 유지): {e}")
        import traceback as _tb
        _tb.print_exc()
    return out_path


_export_korea_excel_with_opm.__opm_model_wrapper__ = True
KoreaDCFModel.export_korea_excel = _export_korea_excel_with_opm
print("[OK] OPM_Model 패치 등록 완료 (Phase2_LongTerm 미드사이클 동기화 + OPM_Model 시트 → 총 11-sheet)")


[OK] OPM_Model 패치 등록 완료 (Phase2_LongTerm 미드사이클 동기화 + OPM_Model 시트 → 총 11-sheet)


## Cell 9 · 실행 — 종목 리스트 → Excel 일괄 출력 (11-sheet: NWC_Detail · OPM_Model 포함)

출력 파일명 끝에 측정일자가 붙습니다 (예: `A005930_..._20260707.xlsx`).

In [23]:
# ═══════════════════════════════════════════════════════════════
#  실행 · EXPORT_TICKERS 리스트 → 종목별 11-sheet Excel 출력 (NWC_Detail · OPM_Model 포함)
#  - 사용자 입력 없음 (대상 종목은 Cell 2 의 EXPORT_TICKERS 에서 지정)
#  - 시각화 없음: run → Excel 저장 → 요약 한 줄 출력의 단순 흐름
#  - SAVE_TO_DB=True 면 DB(TABLE_RESULT)에도 저장
# ═══════════════════════════════════════════════════════════════

assert hasattr(KoreaDCFModel, "export_korea_excel"), \
    "KoreaDCFModel.export_korea_excel 없음 — 바로 위 Excel Export 모듈 등록 셀을 먼저 실행하세요."

run_date = datetime.now().strftime("%Y-%m-%d")
_summary_rows = []
t0 = time.time()

print(f"\n{'='*70}")
print(f"[Individual FCFF Valuation] 대상 {len(EXPORT_TICKERS)}종목  "
      f"run_date={run_date}  SAVE_TO_DB={SAVE_TO_DB}")
print(f"{'='*70}")

for i, _tk in enumerate(EXPORT_TICKERS, 1):
    tk = to_dg_ticker(_tk)
    print(f"\n[{i}/{len(EXPORT_TICKERS)}] {tk} " + "─" * 50)
    try:
        _model = KoreaDCFModel(
            ticker=tk, engine=engine, db_info=db_info,
            rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX,
            verbose=VERBOSE,
        )
        _model.run()
        _path = Path(_model.export_korea_excel(out_dir=EXPORT_DIR))

        # ── 파일명 끝에 FCFF 측정일자 태그 부착 ──────────────
        #    예: A005930_FCFF.xlsx → A005930_FCFF_20260707.xlsx
        #    (모듈이 이미 날짜를 붙여주는 경우엔 중복 방지 위해 생략)
        _date_tag = run_date.replace("-", "")          # 'YYYY-MM-DD' → 'YYYYMMDD'
        if _date_tag not in _path.stem:
            _new_path = _path.with_name(f"{_path.stem}_{_date_tag}{_path.suffix}")
            if _new_path.exists():                      # 같은 날 재실행 → 덮어쓰기
                _new_path.unlink()
            _path = _path.rename(_new_path)

        _rows = _model.save_to_db(run_date) if SAVE_TO_DB else 0

        v = _model.valuation
        tp_str = f"{v['target_price']:,.0f}원" if not np.isnan(v["target_price"]) else "N/A"
        cp_str = f"{v['current_price']:,.0f}원" if v["current_price"] else "N/A"
        up_str = f"{v['upside_pct']:+.1f}%" if not np.isnan(v["upside_pct"]) else "N/A"

        print(f"  ✓ Saved: {_path}")
        print(f"    적정주가   : {tp_str}    현재가: {cp_str}    Upside: {up_str}")
        print(f"    WACC       : {v['wacc']*100:.2f}%   g_terminal: {v['g_terminal']*100:.2f}%")
        print(f"    Moat       : {v['moat_label']}  (ρ={v['moat_rho']:.3f}, "
              f"Phase2={v['n_phase2']}년, g₀ method={v.get('g0_method','n/a')})")
        _oi = v.get("opm_info", {}) or {}
        _mc = v.get("midcycle", {}) or {}
        _opm_fc = _model.result_df["opm_forecast"]
        print(f"    OPM        : {_oi.get('mode_used')}[{_oi.get('reg_method', '-')}] "
              f"R²={_oi.get('reg_r2', float('nan')):.2f}  8Q OPM {_opm_fc.iloc[0]:.1%} → {_opm_fc.iloc[-1]:.1%}")
        if _mc.get("applied"):
            print(f"    Mid-cycle  : OPM {_mc['opm_end']:.1%} → {_mc['opm_mid']:.1%} "
                  f"({_mc['mid_stat']}, {_mc['years']}년 수렴)")
        else:
            print(f"    Mid-cycle  : 미적용 ({_mc.get('reason')})")
        if SAVE_TO_DB:
            print(f"    DB 저장    : {_rows}행 → {TABLE_RESULT}")

        _summary_rows.append({
            "ticker": tk, "status": "ok", "excel": str(_path),
            "target_price": v["target_price"], "current_price": v["current_price"],
            "upside_pct": v["upside_pct"], "wacc": v["wacc"],
            "g_terminal": v["g_terminal"], "moat": v["moat_label"],
            "opm_method": f"{_oi.get('mode_used')}[{_oi.get('reg_method', '-')}]",
            "opm_8q_avg": float(_opm_fc.mean()),
            "opm_mid": _mc.get("opm_mid") if _mc.get("applied") else np.nan,
        })
    except Exception as e:
        print(f"  ✗ FAIL: {e}")
        traceback.print_exc()
        _summary_rows.append({
            "ticker": tk, "status": "fail", "excel": "",
            "target_price": np.nan, "current_price": np.nan,
            "upside_pct": np.nan, "wacc": np.nan,
            "g_terminal": np.nan, "moat": "?",
        })
    finally:
        clear_memory()

# ── 최종 요약 테이블 ─────────────────────────────────────────
elapsed = time.time() - t0
print(f"\n{'='*70}")
print(f"[완료] {len(_summary_rows)}종목  "
      f"OK={sum(r['status']=='ok' for r in _summary_rows)}  "
      f"FAIL={sum(r['status']=='fail' for r in _summary_rows)}  "
      f"경과={elapsed:.0f}s")
print(f"{'='*70}")
summary_df = pd.DataFrame(_summary_rows).set_index("ticker")
display(summary_df)



[Individual FCFF Valuation] 대상 1종목  run_date=2026-09-29  SAVE_TO_DB=False

[1/1] A000660 ──────────────────────────────────────────────────
[01:02:19][A000660] Sales actual=67Q forecast=8Q model=Ensemble created_at=2026-09-29
[01:02:19][A000660] FS wide  shape=(68, 35)  기간=2009-12-31 ~ 2026-09-30
[01:02:19][A000660] [OPM] 회귀[diff] α=-0.93조 β=0.915 R²=1.00 n=20 홀드아웃MAE(조)={'level': 3.154, 'diff': 2.224} → OPM 78.8% … 86.3% (clip [-76.9%,86.3%])
[01:02:19][A000660] ⚠️  v7 영업부채 항목 없음 → legacy proxy 사용
[01:02:19][A000660] [NWC seed] γ×sales=63,358.6B vs 실측=0.0B 괴리 6335862750730939.0%
[01:02:21][A000660] β_raw=1.458 β_blume=1.307  Re=13.6869%  (n=2447, R²=0.56)
[01:02:26][A000660] WACC=13.6869%  Re=13.6869% Rd=4.5390% tax=19.7072%  E/V=100.00% D/V=0.00%  E=1598.59조 D=0.00조
[01:02:26][A000660] EVA: ROIC=44.43%  WACC=13.69%  spread=+30.75%  n_pos(20Q)=10/20
[01:02:26][A000660] Moat: [Some moat]  spread=+30.75%  n_pos=10/20  → ρ=0.75  Phase2=8yr
[01:02:26][A000660] ρ 보정: EVA기반=0.575  OLS=0.40

,status,excel,target_price,current_price,upside_pct,wacc,g_terminal,moat,opm_method,opm_8q_avg,opm_mid
ticker,,,,,,,,,,,
A000660,ok,C:\reports\A000660_FCFF_Korea_v2_20260929.xlsx,2.650438e+06,1693000.0,56.552767,0.136869,0.04,Some moat,regression[diff],0.836911,0.295144
